# SIME — Conciliação MES × SAP em um banco rastreável

**Luan César de Souza Nascimento** · estudante de Engenharia de Controle e Automação (UFPE) · 12/10/2026

Este notebook valida a planilha do case e compara a v3 com o novo método candidato `sp2-evidence-v1`, que registra decisões para MES e SAP, evidências, alternativas e hipóteses revisáveis. Nenhum método confirma o evento físico nem mede acurácia sem revisão humana. O modo padrão usa dados fictícios: basta **Ambiente de execução → Executar tudo**.

### Índice
1. [Objetivo](#objetivo)
2. [Ambiente e módulos](#ambiente)
3. [Entrada](#entrada)
4. [Validação](#validacao)
5. [Associação](#associacao)
6. [Banco SQLite](#banco)
7. [Consultas](#consultas)
8. [Painel de parâmetros](#painel)
9. [Testes](#testes)
10. [Entrega](#entrega)
11. [Limites](#limites)

## 1. Objetivo <a id="objetivo" name="objetivo"></a>

`planilha .xlsx` → validação → normalização → associação → SQLite → consultas e painel

Cada registro guarda arquivo, aba, linha e hash da origem.

## 2. Ambiente e módulos <a id="ambiente" name="ambiente"></a>

Os módulos usados pelos dois métodos estão embutidos na célula recolhida abaixo, sem download externo. A célula seguinte grava cada um e confere o SHA-256, que prova a identidade dos bytes, não a qualidade dos dados.

In [ ]:
import copy
import hashlib
import importlib.util
import json
import os
import shutil
import sqlite3
import subprocess
import sys
import zipfile
from datetime import datetime, time, timedelta
from pathlib import Path
from uuid import uuid4


def instalar_se_faltar(pacote, versao):
    if importlib.util.find_spec(pacote) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', f'{pacote}=={versao}'])


instalar_se_faltar('openpyxl', '3.1.5')
instalar_se_faltar('ipywidgets', '8.1.7')

import ipywidgets as widgets
import openpyxl
import pandas as pd
from IPython.display import display


def esta_no_colab():
    try:
        return importlib.util.find_spec('google.colab') is not None
    except ModuleNotFoundError:
        return False


EM_COLAB = esta_no_colab()
PASTA = Path('/content/SIME') if EM_COLAB else Path.cwd() / 'SIME_colab'
PASTA.mkdir(parents=True, exist_ok=True)
print('Python', sys.version.split()[0], '| openpyxl', openpyxl.__version__, '| saídas em', PASTA)

In [ ]:
# @title Código-fonte dos módulos do projeto (recolhido; abra para inspecionar)
# Cópia literal dos arquivos; SOURCE_HASHES foi calculado sobre os bytes originais.
SOURCE_FILES = {
 "04_dados/auditoria/gpt/GPT-DADOS-01/auditar_fontes.py": "\"\"\"Extração auditada e reproduzível das bases MES e SAP a partir da planilha original.\"\"\"\r\nfrom pathlib import Path\r\nfrom collections import Counter, defaultdict\r\nfrom datetime import datetime, time, timedelta\r\nimport csv\r\nimport hashlib\r\nimport json\r\nimport re\r\nfrom statistics import median\r\nimport unicodedata\r\n\r\nfrom openpyxl import load_workbook\r\nfrom openpyxl.utils.datetime import from_excel\r\n\r\nROOT = Path(__file__).resolve().parents[4]\r\nOUT = Path(__file__).resolve().parent\r\nFILE = ROOT / \"Base de dados Moura - COMPET - Manutencao Inteligente.xlsx\"\r\n\r\n\r\ndef txt(v):\r\n    return re.sub(r\"\\s+\", \" \", str(v or \"\")).strip()\r\n\r\n\r\ndef serial(v):\r\n    return v.isoformat() if isinstance(v, (datetime, time)) else v\r\n\r\n\r\ndef date(v, epoch):\r\n    if v is None: return None\r\n    if isinstance(v, datetime): return v\r\n    if isinstance(v, (int, float)): return from_excel(v, epoch)\r\n    try: return datetime.fromisoformat(str(v))\r\n    except ValueError: return None\r\n\r\n\r\ndef dt(day, clock, epoch):\r\n    day = date(day, epoch)\r\n    if not isinstance(day, datetime): return None\r\n    if isinstance(clock, datetime): clock = clock.time()\r\n    if isinstance(clock, (int, float)):\r\n        clock = (datetime.min + timedelta(days=clock)).time()\r\n    if isinstance(clock, str):\r\n        try: clock = time.fromisoformat(clock)\r\n        except ValueError: return None\r\n    if not isinstance(clock, time): return None\r\n    return datetime.combine(day.date(), clock)\r\n\r\n\r\ndef dump(name, value):\r\n    (OUT / name).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=serial), encoding=\"utf-8\")\r\n\r\n\r\ndef write_csv(name, rows):\r\n    keys = list(dict.fromkeys(k for row in rows for k in row))\r\n    with (OUT / name).open(\"w\", encoding=\"utf-8-sig\", newline=\"\") as f:\r\n        writer = csv.DictWriter(f, fieldnames=keys)\r\n        writer.writeheader(); writer.writerows(rows)\r\n\r\n\r\ndef main():\r\n    OUT.mkdir(parents=True, exist_ok=True)\r\n    digest = hashlib.sha256(FILE.read_bytes()).hexdigest()\r\n    w = load_workbook(FILE, read_only=True, data_only=True)\r\n    formula_w = load_workbook(FILE, read_only=True, data_only=False)\r\n    inventory = {\"arquivo\": FILE.name, \"sha256\": digest, \"excel_epoch\": w.epoch.isoformat(),\r\n        \"abas\": []}\r\n    tables = {}\r\n    quality = {}\r\n    dictionary = []\r\n    for sheet in w:\r\n        headers = [str(v) if v is not None else f\"coluna_{i+1}\" for i,v in enumerate(next(sheet.iter_rows(values_only=True)))]\r\n        rows = []\r\n        for line, values in enumerate(sheet.iter_rows(min_row=2, values_only=True), 2):\r\n            if not any(v is not None for v in values): continue\r\n            rows.append({\"_arquivo\": FILE.name, \"_aba\": sheet.title, \"_linha_excel\": line,\r\n                \"_id_auditoria\": (\"MES\" if sheet.title.startswith(\"MES\") else \"SAP\") + f\"-R{line:04d}\",\r\n                **{key: serial(value) for key,value in zip(headers, values)}})\r\n        formulas = [{\"celula\": c.coordinate, \"formula\": c.value} for r in formula_w[sheet.title] for c in r if c.data_type == \"f\"]\r\n        errors = [{\"celula\": c.coordinate, \"valor\": c.value} for r in sheet for c in r if c.data_type == \"e\"]\r\n        columns = {}\r\n        for h in headers:\r\n            vals = [r[h] for r in rows]\r\n            missing = sum(v is None or (isinstance(v,str) and not v.strip()) for v in vals)\r\n            counter = Counter(str(v) for v in vals if v is not None)\r\n            columns[h] = {\"ausentes\": missing, \"denominador\": len(rows), \"distintos\": len(counter),\r\n                \"tipos_exportados\": dict(Counter(type(v).__name__ for v in vals)), \"exemplo\": next((v for v in vals if v is not None),None)}\r\n            dictionary.append(f\"| {sheet.title} | `{h}` | {','.join(columns[h]['tipos_exportados'])} | {missing}/{len(rows)} | `{str(columns[h]['exemplo'])[:95]}` |\")\r\n        full = Counter(tuple(str(r[h]) for h in headers) for r in rows)\r\n        duplicates = [[r['_id_auditoria'] for r in rows if tuple(str(r[h]) for h in headers)==k]\r\n                      for k,n in full.items() if n>1]\r\n        quality[sheet.title] = {\"registros\": len(rows), \"colunas\": columns, \"grupos_duplicatas_exatas\": duplicates,\r\n            \"formulas\": len(formulas), \"erros_excel\": errors}\r\n        tables[sheet.title] = rows\r\n        inventory['abas'].append({\"aba\": sheet.title, \"cabecalho_linha\":1, \"registros\": len(rows), \"cabecalhos_originais\":headers,\r\n            \"csv\": \"MES.csv\" if sheet.title.startswith(\"MES\") else \"SAP.csv\", \"formulas\":formulas})\r\n    mes = tables['MES - PRODUÇÃO']; sap = tables['SAP - MANUTENÇÃO']\r\n    mes_issues = defaultdict(list); mes_groups=Counter()\r\n    for r in mes:\r\n        r['maquina_norm'] = txt(r['MÁQUINA ']); r['equipamento_norm'] = txt(r['EQUIPAMENTO'])\r\n        r['componente_norm'] = txt(r['Componente']); r['familia_norm'] = txt(r['Famila'])\r\n        start=date(r['Inicio'], w.epoch); end=date(r['Término'], w.epoch); day=date(r['Data'],w.epoch)\r\n        r['inicio_iso']=serial(start);r['fim_iso']=serial(end)\r\n        r['duracao_derivada_h']=(end-start).total_seconds()/3600 if start and end else None\r\n        if start and day and start.date()!=day.date():mes_issues['data_diferente_inicio'].append(r['_id_auditoria'])\r\n        if start and end and start.date()!=end.date():mes_issues['vira_dia'].append(r['_id_auditoria'])\r\n        if r['duracao_derivada_h'] is not None and r['duracao_derivada_h']<0:mes_issues['duracao_negativa'].append(r['_id_auditoria'])\r\n        if r['Horas'] is None:mes_issues['horas_ausentes'].append(r['_id_auditoria'])\r\n        elif r['duracao_derivada_h'] is not None and abs(float(r['Horas'])-r['duracao_derivada_h'])*3600>1:mes_issues['diferenca_duracao_mais_1s'].append(r['_id_auditoria'])\r\n        if r['Componente'] != r['componente_norm']:mes_issues['espacos_componente'].append(r['_id_auditoria'])\r\n        mes_groups[(r['maquina_norm'],r['familia_norm'],r['equipamento_norm'],r['componente_norm'])]+=1\r\n    sap_issues=defaultdict(list)\r\n    for r in sap:\r\n        location=txt(r['Loc.instalação']);match=re.search(r'\\.MAQ([A-D])(?:\\.|$)',location)\r\n        r['maquina_inferida']=match.group(1) if match else None\r\n        r['mapeamento_maquina_status']='inferido de segmento MAQ; validar tecnicamente'\r\n        start=dt(r['Dta.início real'],r['InícReal hr.'],w.epoch); end=dt(r['Dt.real fim'],r['Fim real'],w.epoch)\r\n        base=date(r['Data-base iníc.'],w.epoch)\r\n        r['inicio_iso']=serial(start);r['fim_iso']=serial(end);r['data_base_iso']=serial(base)\r\n        r['duracao_intervalo_registrado_h']=(end-start).total_seconds()/3600 if start and end else None\r\n        if not start or not end:sap_issues['inicio_ou_fim_invalido'].append(r['_id_auditoria'])\r\n        if r['duracao_intervalo_registrado_h'] is not None and r['duracao_intervalo_registrado_h']<0:sap_issues['intervalo_negativo'].append(r['_id_auditoria'])\r\n        if start and end and start.date()!=end.date():sap_issues['vira_dia'].append(r['_id_auditoria'])\r\n        if start and base and start.date()!=base.date():sap_issues['data_base_diferente_inicio'].append(r['_id_auditoria'])\r\n        if not match:sap_issues['maquina_nao_mapeada'].append(r['_id_auditoria'])\r\n    overlap=[]\r\n    for machine in sorted({r['maquina_norm'] for r in mes}):\r\n        group=sorted([r for r in mes if r['maquina_norm']==machine],key=lambda r:r['inicio_iso'] or '')\r\n        active=[]\r\n        for row in group:\r\n            start=row['inicio_iso'];active=[a for a in active if a['fim_iso'] and start and a['fim_iso']>start]\r\n            for a in active:overlap.append({'maquina':machine,'id_1':a['_id_auditoria'],'id_2':row['_id_auditoria']})\r\n            active.append(row)\r\n    quality['MES - PRODUÇÃO']['verificacoes_temporais_e_textuais']=dict(mes_issues)\r\n    quality['MES - PRODUÇÃO']['pares_sobrepostos_mesma_maquina']=overlap\r\n    quality['SAP - MANUTENÇÃO']['verificacoes_temporais']=dict(sap_issues)\r\n    orders=Counter(str(r['N. Ordem']) for r in sap)\r\n    quality['SAP - MANUTENÇÃO']['ordens_repetidas']={k:n for k,n in orders.items() if n>1}\r\n    summary = {'fonte_sha256':digest,\r\n        'registros':{'MES':len(mes),'SAP':len(sap)},\r\n        'MES_maquinas':dict(Counter(r['maquina_norm'] for r in mes)),\r\n        'SAP_maquinas_inferidas':dict(Counter(r['maquina_inferida'] for r in sap)),\r\n        'MES_familias':dict(Counter(r['familia_norm'] for r in mes)),\r\n        'SAP_tipos':dict(Counter(r['Tipo'] for r in sap)),\r\n        'SAP_atividades':dict(Counter(str(r['Tip.ativ.manut.']) for r in sap)),\r\n        'MES_problemas':{k:len(v) for k,v in mes_issues.items()},\r\n        'SAP_problemas':{k:len(v) for k,v in sap_issues.items()},\r\n        'MES_pares_sobrepostos':dict(Counter(r['maquina'] for r in overlap)),\r\n        'cobertura':{'MES':[min(r['inicio_iso'] for r in mes),max(r['fim_iso'] for r in mes)],\r\n            'SAP':[min(r['inicio_iso'] for r in sap if r['inicio_iso']),max(r['fim_iso'] for r in sap if r['fim_iso'])]},\r\n        'recorrencias_A': [{'maquina':a,'familia':b,'equipamento':c,'componente':d,'contagem':n}\r\n            for (a,b,c,d),n in mes_groups.most_common() if a=='A'],\r\n        'limites':['maquina SAP inferida, nao confirmada tecnicamente','códigos de componente sem dicionario tecnico',\r\n            'exposicao/horas operadas, marcos de chamados, custos e HH nao presentes nas colunas',\r\n            'duracao SAP e intervalo entre marcos cadastrados, nao tempo de reparo validado']}\r\n    sample = sorted([r for r in sap if r['maquina_inferida']=='A'],key=lambda r:r['_linha_excel'])\r\n    # Amostra deterministica de inspeção, incluindo diversidade de tipos/atividades; nao e verdade rotulada.\r\n    buckets=defaultdict(list)\r\n    for r in sample:buckets[(r['Tipo'],r['Tip.ativ.manut.'])].append(r)\r\n    selected=[]\r\n    for rows in buckets.values():\r\n        for index in sorted({0,len(rows)//2,len(rows)-1}):selected.append(rows[index])\r\n    selected=sorted({r['_id_auditoria']:r for r in selected}.values(),key=lambda r:r['_linha_excel'])\r\n    write_csv('MES.csv',mes);write_csv('SAP.csv',sap);write_csv('AMOSTRA_SAP_A.csv',selected)\r\n    dump('inventario.json',inventory);dump('perfil_qualidade.json',quality);dump('resumo.json',summary)\r\n    dictionary_text='# Dicionário observado dos dados\\n\\nTipos referem-se a exportação; datas SAP numéricas são seriais Excel. Linhas originais e cabeçalhos preservados.\\n\\n'\r\n    dictionary_text+='| Aba | Campo original | Tipos exportados | Ausentes/total | Exemplo |\\n|---|---|---|---|---|\\n'+'\\n'.join(dictionary)\r\n    dictionary_text+='\\n\\nMES: Horas é duração informada da ocorrência, comparada com fim–início; Data pode representar dia operacional distinto da data do início. EQUIPAMENTO é a subdivisão documentada no enunciado; códigos de componente exigem dicionário técnico.\\n\\nSAP: datas numéricas convertidas com epoch do arquivo; intervalo início–fim não é HH ou tempo de reparo confirmado. N. Ordem identifica uma ordem, cujo número de linhas é auditado. MAQA etc. sugere máquina, com mapeamento inferido. Tipo Corretiva Planejada não é convertido em Preventiva. Não há marcos de solicitação/aceite, exposição operacional, custos ou conteúdo de plano preventivo nas colunas.\\n'\r\n    (OUT/'DICIONARIO_DADOS.md').write_text(dictionary_text,encoding='utf-8')\r\n    (OUT/'DIAGNOSTICO_BASES.md').write_text('# Diagnóstico reproduzível\\n\\nEvidências numéricas em resumo.json e perfil_qualidade.json.\\n\\n```json\\n'+json.dumps(summary,ensure_ascii=False,indent=2)+'\\n```\\n\\nNão foram excluídas linhas nem corrigidos horários. Normalizações são derivadas e conservam as colunas originais. Detecção de sobreposição usa intervalos da mesma máquina, sem declarar erro ou somar durações como perda exclusiva.\\n',encoding='utf-8')\r\n    w.close();formula_w.close()\r\n    assert hashlib.sha256(FILE.read_bytes()).hexdigest()==digest\r\n    print(json.dumps(summary,ensure_ascii=False,indent=2))\r\n\r\n\r\nif __name__=='__main__':main()\r\n",
 "05_subprodutos/gerar_historico_mvp.py": "\"\"\"Derivados completos, regras exploratórias e baseline textual do MVP. Executor: integrador.\"\"\"\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nfrom collections import Counter, defaultdict\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport re\nimport statistics\nimport unicodedata\nfrom importlib.util import spec_from_file_location, module_from_spec\n\nROOT = Path(__file__).resolve().parents[1]\n_association_path = Path(__file__).parent/'02_conciliacao/associacao_automatica.py'\n_association_spec = spec_from_file_location('moura_associacao_automatica', _association_path)\n_association_module = module_from_spec(_association_spec)\n_association_spec.loader.exec_module(_association_module)\nINPUT = ROOT/'04_dados/auditoria/gpt/GPT-DADOS-01'\nASSET_CATALOG = ROOT/'05_subprodutos/catalogo_ativos.json'\nVERSION = 'mvp-1.2'\nDURATION_COMPARISON = {\n    'method': 'duration-v1',\n    'absoluteThresholdMinutes': 15,\n    'relativeThreshold': 0.25,\n    'rule': 'Relevant when absolute difference >= 15 minutes AND absolute difference / max(MES duration, SAP duration) >= 0.25.',\n    'differenceDirection': 'SAP duration minus MES duration; absolute and ratio fields are non-negative.',\n    'zeroDurations': 'If both durations are zero, ratio is 0.',\n    'incompleteDurations': 'If either duration is unavailable, difference, ratio, and relevance are null.',\n    'justification': 'Proposed screening rule requiring both a material absolute gap and a substantial relative gap; not calibrated or human validated.',\n    'validated': False,\n    'notEquivalentTo': ['confirmed event association', 'MTTR', 'labor hours (HH)'],\n}\n\n\ndef normal(value):\n    return ' '.join(''.join(c for c in unicodedata.normalize('NFKD',value.upper())\n                           if not unicodedata.combining(c)).split())\n\n\ndef read(name):\n    with (INPUT/name).open(encoding='utf-8-sig',newline='') as stream:\n        return list(csv.DictReader(stream))\n\n\ndef source(row):\n    inventory=json.loads((ROOT/'01_fontes/inventario.json').read_text(encoding='utf-8'))['fontes']\n    return dict(file=row['_arquivo'],sheet=row['_aba'],row=int(row['_linha_excel']),\n        hash=next(r['sha256'] for r in inventory if r['arquivo']==row['_arquivo']))\n\n\ndef load_assets(path=ASSET_CATALOG):\n    \"\"\"Load a data-driven asset catalog; business rules use IDs, never specific letters.\"\"\"\n    data=json.loads(Path(path).read_text(encoding='utf-8-sig'))\n    assets=data.get('assets') if isinstance(data,dict) else None\n    if not isinstance(assets,list) or not assets:\n        raise ValueError('Catálogo deve conter uma lista assets não vazia.')\n    ids=set();source_keys=set();clean=[]\n    for asset in assets:\n        required={'id','sourceKey','displayName','assetNature','studyFocus'}\n        if not isinstance(asset,dict) or not required.issubset(asset):\n            raise ValueError('Cada ativo deve conter id, sourceKey, displayName, assetNature e studyFocus.')\n        if not all(isinstance(asset[k],str) and asset[k].strip() for k in ('id','sourceKey','displayName','assetNature')):\n            raise ValueError('Campos textuais do catálogo não podem ser vazios.')\n        if not re.fullmatch(r'[A-Za-z0-9_-]{1,64}',asset['id']):\n            raise ValueError('id deve seguir o formato aceito pelo catálogo do aplicativo.')\n        if not isinstance(asset['studyFocus'],bool):\n            raise ValueError('studyFocus deve ser booleano.')\n        if asset['id'] in ids or asset['sourceKey'] in source_keys:\n            raise ValueError('IDs e sourceKey do catálogo devem ser únicos.')\n        ids.add(asset['id']);source_keys.add(asset['sourceKey'])\n        clean.append({k:asset[k] for k in ('id','sourceKey','displayName','assetNature','studyFocus')})\n    return clean\n\n\ndef resolve_asset(source_key, assets):\n    matches=[a for a in assets if a['sourceKey']==source_key]\n    if len(matches)!=1:\n        raise ValueError(f'Ativo de origem sem mapeamento único no catálogo: {source_key!r}')\n    return matches[0]\n\n\ndef duration_divergence(mes_duration, sap_duration):\n    \"\"\"Return proposed duration-v1 comparison; it does not confirm event identity.\"\"\"\n    if mes_duration is None or sap_duration is None:\n        difference=absolute=ratio=relevant=None\n    else:\n        difference=round(float(sap_duration)-float(mes_duration),6)\n        absolute=round(abs(difference),6)\n        denominator=max(float(mes_duration),float(sap_duration))\n        exact_ratio=absolute/denominator if denominator else 0.0\n        ratio=round(exact_ratio,6)\n        relevant=(absolute>=DURATION_COMPARISON['absoluteThresholdMinutes'] and\n                  exact_ratio>=DURATION_COMPARISON['relativeThreshold'])\n    return {\n        'durationDifferenceMinutes':difference,\n        'durationDifferenceAbsoluteMinutes':absolute,\n        'durationDifferenceRatio':ratio,\n        'durationDifferenceRelevant':relevant,\n        'durationDifferenceMethod':DURATION_COMPARISON['method'],\n    }\n\n\ndef minutes(start,end):\n    if not start or not end:\n        return None\n    value=(datetime.fromisoformat(end)-datetime.fromisoformat(start)).total_seconds()/60\n    return round(value,6) if value>=0 else None\n\n\ndef field(text,pattern,label=None):\n    match=re.search(pattern,text,re.I)\n    if not match:\n        return dict(status='insufficient',value=None,evidence=None)\n    prefix=normal(text[max(0,match.start()-25):match.start()])\n    if re.search(r'\\b(?:NAO|SEM|NUNCA)(?:\\s+(?:FOI|ESTA|ESTAVA))?\\s*$',prefix):\n        return dict(status='insufficient',value=None,evidence=None)\n    return dict(status='supported',value=label or normal(match.group()),evidence=match.group())\n\n\ndef classify(record):\n    text=record['text']\n    component=field(text,r'\\b(?:LONAS?|DISCOS?|ROLOS?|MOTORES?|MOTOR|CORREIAS?|SENSORES?|SENSOR|FORNOS?|ROLAMENTOS?|CORRENTES?|BOMBAS?|BOMBA|V[ÁA]LVULAS?|CILINDROS?|REDUTORES?|EIXOS?|ESTEIRAS?|PAIN[ÉE]IS|PAINEL)\\b')\n    symptom=field(text,r'\\b(?:TRAVAD[AO]S?|ENTUPID[AO]S?|EM FALHA|FALHAND[AO]|VAZAMENTOS?|VAZANDO|N[ÃA]O (?:LIGA|FUNCIONA|ACIONA|ABRE|FECHA)|SEM (?:PRESS[ÃA]O|AQUECIMENTO|SINAL)|SUPERAQUECIMENTO|RU[ÍI]DOS?|VIBRA[ÇC][ÃA]O|DESALINHAD[AO]S?)\\b')\n    mode=field(text,r'\\b(?:QUEBRAD[AO]S?|RASGAD[AO]S?|ROMPID[AO]S?|DESGASTAD[AO]S?|TRINCAD[AO]S?)\\b')\n    requested=field(text,r'\\b(?:SUBSTITUIR|TROCAR|ALINHAR|AJUSTAR|VERIFICAR|CORRIGIR|LIMPAR|INSTALAR|INSPECIONAR|REGULAR|REPARAR|LUBRIFICAR)\\b')\n    reported=field(text,r'\\b(?:TROCAD[AO]S?|SUBSTITU[ÍI]D[AO]S?|AJUSTAD[AO]S?|ALINHAD[AO]S?|INSTALAD[AO]S?|LIMP[AO]S?|CORRIGID[AO]S?|REPARAD[AO]S?)\\b')\n    mentioned=field(text,r'\\b(?:CORRE[ÇC][ÃA]O|AJUSTE|SUBSTITUI[ÇC][ÃA]O|TROCA|ALINHAMENTO|LIMPEZA|REPARO|LUBRIFICA[ÇC][ÃA]O)\\b')\n    unknown=dict(status='insufficient',value=None,evidence=None)\n    questions=[]\n    if component['status']=='insufficient':\n        questions.append('Qual componente foi observado? Informe a identificação sem expandir siglas desconhecidas.')\n    if symptom['status']=='insufficient' and mode['status']=='insufficient':\n        questions.append('O que foi observado e em qual condição de operação?')\n    questions += ['O que foi efetivamente feito durante o atendimento?',\n                  'Que evidência foi registrada para confirmar o retorno à operação?',\n                  'Qual informação falta para investigar a causa?']\n    return dict(sapId=record['id'],method='baseline lexical determinística, sem LLM em tempo de uso',\n        taxonomyVersion=VERSION,component=component,symptom=symptom,failureMode=mode,\n        causeHypothesis=unknown.copy(),causeConfirmed=unknown.copy(),actionRequested=requested,\n        actionMentioned=mentioned,actionReported=reported,\n        actionPerformed=dict(status='unverified',value=None,evidence=None),\n        questions=questions,quality=dict(wordCount=len(text.split()),characterCount=len(text),\n        note='Trechos sustentam apenas categorias textuais; não confirmam execução ou causa.'))\n\n\ndef candidates(mes,sap):\n    complete=defaultdict(list)\n    for row in sap:\n        if row['durationMinutes'] is not None:\n            complete[row['machine']].append((row,datetime.fromisoformat(row['start']),datetime.fromisoformat(row['end'])))\n    edges=[]\n    for m in mes:\n        ms,me=datetime.fromisoformat(m['start']),datetime.fromisoformat(m['end'])\n        for s,ss,se in complete[m['machine']]:\n            gap=max(0,(ss-me).total_seconds()/60,(ms-se).total_seconds()/60)\n            if gap>60:\n                continue\n            overlap=max(0,(min(me,se)-max(ms,ss)).total_seconds()/60)\n            exact=normal(m['subdivision'])==normal(s['subdivision'])\n            for rule in ['machine']+(['text'] if exact else []):\n                for tolerance in [0,15,60]:\n                    if gap<=tolerance:\n                        edge=dict(rule=rule,tolerance=tolerance,mesId=m['id'],sapId=s['id'],\n                            machine=m['machine'],gapMinutes=round(gap,6),overlapMinutes=round(overlap,6),\n                            startDeltaMinutes=round((ss-ms).total_seconds()/60,6),textEqual=exact,\n                            status='candidate')\n                        edge.update(duration_divergence(m.get('durationMinutes'),s.get('durationMinutes')))\n                        edges.append(edge)\n    cm=Counter((e['rule'],e['tolerance'],e['mesId']) for e in edges)\n    cs=Counter((e['rule'],e['tolerance'],e['sapId']) for e in edges)\n    for e in edges:\n        e['mesCandidateCount']=cm[e['rule'],e['tolerance'],e['mesId']]\n        e['sapCandidateCount']=cs[e['rule'],e['tolerance'],e['sapId']]\n    scenarios=[]\n    for machine in sorted({r['machine'] for r in mes}):\n        subset=[r for r in mes if r['machine']==machine]\n        for rule in ['machine','text']:\n            for tolerance in [0,15,60]:\n                current=[e for e in edges if (e['machine'],e['rule'],e['tolerance'])==(machine,rule,tolerance)]\n                counts=Counter(e['mesId'] for e in current)\n                scenarios.append(dict(machine=machine,rule=rule,tolerance=tolerance,pairs=len(current),\n                    mesCount=len(subset),noCandidate=len(subset)-len(counts),\n                    oneCandidate=sum(v==1 for v in counts.values()),multipleCandidates=sum(v>1 for v in counts.values())))\n    return edges,scenarios\n\n\ndef calculate(mes,sap,assets):\n    asset_by_id={a['id']:a for a in assets}\n    summaries=[]\n    groups=defaultdict(list);months=defaultdict(list)\n    for r in mes:\n        groups[r['machine'],r['family'],r['subdivision'],r['component']].append(r)\n        months[r['machine'],r['start'][:7]].append(r)\n    for machine in sorted({r['machine'] for r in mes}):\n        records=[r for r in mes if r['machine']==machine]\n        asset=asset_by_id[machine]\n        orders=[r for r in sap if r['machine']==machine]\n        breaks=[r for r in records if r['family']=='QUEBRA']\n        summaries.append(dict(machine=machine,sourceKey=asset['sourceKey'],displayName=asset['displayName'],\n            studyFocus=asset['studyFocus'],mesCount=len(records),sapCount=len(orders),breakCount=len(breaks),\n            adjustmentCount=sum(r['family']=='AJUSTE' for r in records),\n            mesHours=round(sum(r['durationMinutes'] for r in records)/60,6),\n            breakHours=round(sum(r['durationMinutes'] for r in breaks)/60,6),\n            incompleteSap=sum(r['durationMinutes'] is None for r in orders),\n            start=min(r['start'] for r in records),end=max(r['end'] for r in records),\n            assetNature=asset['assetNature']))\n    grouped=[]\n    for (machine,family,subdivision,component),records in sorted(groups.items(),key=lambda item:tuple(v or '' for v in item[0])):\n        asset=asset_by_id[machine]\n        grouped.append(dict(id='|'.join([machine,family,subdivision,component or 'SEM COMPONENTE']),machine=machine,\n            sourceKey=asset['sourceKey'],displayName=asset['displayName'],\n            family=family,subdivision=subdivision,component=component,count=len(records),\n            hours=round(sum(r['durationMinutes'] for r in records)/60,6),\n            medianMinutes=statistics.median(r['durationMinutes'] for r in records),mesIds=[r['id'] for r in records]))\n    monthly=[]\n    for (machine,month),records in sorted(months.items()):\n        breaks=[r for r in records if r['family']=='QUEBRA']\n        monthly.append(dict(machine=machine,month=month,count=len(records),breakCount=len(breaks),\n            hours=round(sum(r['durationMinutes'] for r in records)/60,6),\n            breakHours=round(sum(r['durationMinutes'] for r in breaks)/60,6),\n            coverageNote='Junho tem cobertura parcial; exposição operacional indisponível.' if month=='2026-06'\n                else 'Contagem de registros, sem exposição operacional ou comprovação de falhas físicas.'))\n    return dict(summaries=summaries,groups=grouped,months=monthly,\n        definitions=dict(mesHours='Soma de durações dos registros MES; não indisponibilidade total validada.',\n            breakCount='Registros da família QUEBRA; não episódios físicos independentes confirmados.',\n            mtbf='Não avaliável sem exposição operacional e definição de falha.',\n            mttr='Não avaliável sem marcos de reparo e definição validada.',\n            preventive='Não avaliável: Corretiva Planejada/CP não comprova preventiva.'))\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--output',default='D:/Moura/Site/data/historico.json')\n    parser.add_argument('--assets',default=str(ASSET_CATALOG))\n    parser.add_argument('--report',default=str(ROOT/'06_validacao/geracao-mvp.json'))\n    args=parser.parse_args()\n    assets=load_assets(args.assets)\n    originals=json.loads((ROOT/'01_fontes/inventario.json').read_text(encoding='utf-8'))['fontes']\n    for entry in originals:\n        actual=hashlib.sha256((ROOT/entry['arquivo']).read_bytes()).hexdigest()\n        if actual!=entry['sha256']:\n            raise RuntimeError('Fonte alterada: '+entry['arquivo'])\n    mes=[]\n    for r in read('MES.csv'):\n        asset=resolve_asset(r['maquina_norm'],assets)\n        mes.append(dict(id=r['_id_auditoria'],machine=asset['id'],assetSourceKey=asset['sourceKey'],family=r['familia_norm'],\n            subdivision=r['equipamento_norm'],component=r['componente_norm'] or None,\n            start=r['inicio_iso'],end=r['fim_iso'],durationMinutes=minutes(r['inicio_iso'],r['fim_iso']),\n            recordedHours=float(r['Horas']) if r['Horas'] else None,operationalDate=r['Data'],shift=r['Tur'],\n            source=source(r),raw={k:v for k,v in r.items() if not k.startswith('_')},origin='history',\n            transformations=['Normalização em colunas adicionais; originais preservados.','Intervalo calculado sem conversão de fuso.'],\n            qualityFlags=(['component_missing'] if not r['componente_norm'] else [])+\n                (['operational_day_differs_from_start'] if r['Data'][:10]!=r['inicio_iso'][:10] else [])))\n    sap=[]\n    for r in read('SAP.csv'):\n        asset=resolve_asset(r['maquina_inferida'],assets)\n        sap.append(dict(id=r['_id_auditoria'],orderId=r['N. Ordem'],machine=asset['id'],assetSourceKey=asset['sourceKey'],\n            subdivision=r['Descrição'],type=r['Tipo'],activity=r['Tip.ativ.manut.'],text=r['Texto breve'],\n            start=r['inicio_iso'] or None,end=r['fim_iso'] or None,\n            durationMinutes=minutes(r['inicio_iso'],r['fim_iso']),mappingStatus=r['mapeamento_maquina_status'],\n            source=source(r),raw={k:v for k,v in r.items() if not k.startswith('_')},origin='history',\n            transformations=['Datas Excel convertidas na extração auditada.','Máquina inferida do segmento MAQ; validar mapa técnico.'],\n            qualityFlags=['machine_inferred']+(['interval_incomplete'] if not r['fim_iso'] else [])))\n    edges,scenarios=candidates(mes,sap)\n    classifications=[classify(row) for row in sap]\n    duplicate_count=0\n    for records in [mes,sap]:\n        counts=Counter(json.dumps(r['raw'],sort_keys=True,ensure_ascii=False) for r in records)\n        for record in records:\n            if counts[json.dumps(record['raw'],sort_keys=True,ensure_ascii=False)]>1:\n                record['qualityFlags'].append('possible_exact_duplicate');duplicate_count+=1\n    association=_association_module.associate(mes,sap)\n    result=dict(schemaVersion=1,datasetVersion=VERSION,metadata=dict(version=VERSION,generatedAt=datetime.now(timezone.utc).isoformat(),\n        sourceHashes=originals,pipelineSha256=hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        assets=assets,assetCatalogSha256=hashlib.sha256(Path(args.assets).read_bytes()).hexdigest(),durationComparison=DURATION_COMPARISON,\n        inputHashes={name:hashlib.sha256((INPUT/name).read_bytes()).hexdigest() for name in ['MES.csv','SAP.csv']},\n        mesCount=len(mes),sapCount=len(sap),classificationCount=len(classifications),\n        duplicateCriterion='Igualdade de todos os campos originais/derivados raw, excluindo ID/linha; não prova duplicidade física.',\n        possibleExactDuplicateRows=duplicate_count,\n        historicalTimezone='Não informado: timestamps históricos preservados como horário local sem conversão.',\n        association='Conciliação automática v3; mapeamento de ativo inferido gera sugestão, não vínculo. Score não probabilístico; sem verdade-terreno não se declara acurácia.',\n        associationAlgorithm=association['algorithmVersion'],\n        limitations=['Sem validação humana de pares, códigos ou taxonomia.','Natureza dos ativos vem do catálogo configurado.',\n            'Horas consistentes com intervalos não comprovam definição física.',\n            'Sobreposição em B/C/D pode contar tempo duas vezes; duração somada não é disponibilidade.',\n            'Relações exclusivas não são vínculos confirmados.']),\n        mes=mes,sap=sap,candidates=edges,association=association,classifications=classifications,\n        scenarios=scenarios,metrics=calculate(mes,sap,assets))\n    path=Path(args.output);path.parent.mkdir(parents=True,exist_ok=True)\n    temporary=path.with_suffix('.json.tmp');temporary.write_text(json.dumps(result,ensure_ascii=False,separators=(',',':')),encoding='utf-8');temporary.replace(path)\n    report=Path(args.report);report.parent.mkdir(parents=True,exist_ok=True)\n    report.write_text(json.dumps(dict(output=str(path),sha256=hashlib.sha256(path.read_bytes()).hexdigest(),\n        records=dict(mes=len(mes),sap=len(sap),classifications=len(classifications),candidates=len(edges)),\n        scenarios=scenarios,referenceFocus=next((x for x in result['metrics']['summaries'] if x['studyFocus']),None),\n        assets=assets,durationComparison=DURATION_COMPARISON,sourceHashesPreserved=True),ensure_ascii=False,indent=2),encoding='utf-8')\n    print(json.dumps(dict(output=str(path),mes=len(mes),sap=len(sap),candidates=len(edges),\n        referenceFocus=next((x for x in result['metrics']['summaries'] if x['studyFocus']),None)),ensure_ascii=False))\n\n\nif __name__=='__main__':main()\n",
 "05_subprodutos/02_conciliacao/associacao_automatica.py": "\"\"\"Associação determinística e conservadora entre eventos MES e ordens SAP.\n\nO score ordena candidatos; não é probabilidade nem acurácia. Uma associação\nautomática exige sinais independentes, exclusividade recíproca e um limiar alto.\n\"\"\"\nfrom collections import defaultdict\nfrom datetime import datetime\nimport re\nimport unicodedata\n\n\nALGORITHM_VERSION = 'sp2-association-v3'\nPOLICY = {\n    'candidateWindowMinutes': 60,\n    'autoMinimumScore': 90,\n    'minimumWinnerMargin': 10,\n    'autoRequiresMutualUniqueBest': True,\n    'autoRequiresExactSubdivisionOrComponentText': True,\n    'autoRequiresCompleteSapInterval': True,\n    'autoRequiresValidatedAssetMapping': True,\n    'scoreIsProbability': False,\n    'description': 'Determinístico; ativo compatível, descrição/subdivisão, proximidade dos intervalos e duração. '\n                   'Mapeamento inferido produz sugestão, nunca vínculo. Sem verdade-terreno, score não mede acurácia.'\n}\n\n\ndef normalize(value):\n    value = unicodedata.normalize('NFKD', str(value or '').upper())\n    value = ''.join(char for char in value if not unicodedata.combining(char))\n    return ' '.join(re.findall(r'[A-Z0-9]+', value))\n\n\ndef _dt(value):\n    try:\n        parsed = datetime.fromisoformat(value) if value else None\n    except (TypeError, ValueError) as exc:\n        raise ValueError(f'Timestamp ISO inválido: {value!r}') from exc\n    if parsed and parsed.tzinfo is not None:\n        raise ValueError('Timestamp com fuso não esperado: normalize para a mesma base civil antes da conciliação.')\n    return parsed\n\n\ndef _validate_intervals(mes_records, sap_records):\n    for system, records in (('MES', mes_records), ('SAP', sap_records)):\n        for row in records:\n            start, end = _dt(row.get('start')), _dt(row.get('end'))\n            if start is None or (system == 'MES' and end is None):\n                raise ValueError(f'{system} {row.get(\"id\")} sem início/fim obrigatório para conciliação.')\n            if end is not None and end < start:\n                raise ValueError(f'{system} {row.get(\"id\")} tem fim anterior ao início.')\n\n\ndef _candidate(mes, sap):\n    ms, me = _dt(mes.get('start')), _dt(mes.get('end'))\n    ss, se = _dt(sap.get('start')), _dt(sap.get('end'))\n    if not ms or not me or not ss:\n        return None\n    duration_m = (me - ms).total_seconds() / 60\n    duration_s = (se - ss).total_seconds() / 60 if se else None\n    start_delta = (ss - ms).total_seconds() / 60\n    if se:\n        gap = max(0.0, (ss - me).total_seconds() / 60, (ms - se).total_seconds() / 60)\n        if gap > POLICY['candidateWindowMinutes']:\n            return None\n        overlap = max(0.0, (min(me, se) - max(ms, ss)).total_seconds() / 60)\n    else:\n        # Uma ordem SAP sem fim pode ser inspecionada, mas nunca autoassociada.\n        if start_delta < -POLICY['candidateWindowMinutes'] or start_delta > duration_m + POLICY['candidateWindowMinutes']:\n            return None\n        gap, overlap = None, None\n\n    exact_subdivision = bool(normalize(mes.get('subdivision'))) and (\n        normalize(mes.get('subdivision')) == normalize(sap.get('subdivision')))\n    component = normalize(mes.get('component'))\n    sap_text_tokens = set(normalize(sap.get('text')).split())\n    component_tokens = {token for token in component.split() if len(token) >= 3}\n    shared_component_tokens = sorted(component_tokens & sap_text_tokens)\n\n    end_delta = (se - me).total_seconds() / 60 if se else None\n    duration_delta = duration_s - duration_m if duration_s is not None else None\n    score = 20  # mesmo ativo conforme a extração; origem SAP pode ser inferida\n    if exact_subdivision:\n        score += 25\n    if shared_component_tokens:\n        score += 15\n    if overlap is not None:\n        score += 12 if overlap > 0 else 10\n    elif abs(start_delta) <= 15:\n        score += 5\n    abs_start = abs(start_delta)\n    score += 10 if abs_start <= 5 else 7 if abs_start <= 15 else 3 if abs_start <= 60 else 0\n    if end_delta is not None:\n        abs_end = abs(end_delta)\n        score += 10 if abs_end <= 5 else 7 if abs_end <= 15 else 3 if abs_end <= 60 else 0\n    if duration_delta is not None:\n        abs_duration = abs(duration_delta)\n        max_duration = max(duration_m, duration_s, 0.0)\n        ratio = abs_duration / max_duration if max_duration else 0.0\n        score += 15 if abs_duration <= 5 and ratio <= .10 else 10 if abs_duration <= 15 and ratio <= .25 else 5 if abs_duration <= 60 and ratio <= .50 else 0\n    else:\n        ratio = None\n    return {\n        'mesId': mes['id'], 'sapId': sap['id'], 'machine': mes.get('machine'),\n        'score': min(100, score), 'gapMinutes': round(gap, 3) if gap is not None else None,\n        'overlapMinutes': round(overlap, 3) if overlap is not None else None,\n        'startDeltaMinutes': round(start_delta, 3),\n        'endDeltaMinutes': round(end_delta, 3) if end_delta is not None else None,\n        'durationDifferenceMinutes': round(duration_delta, 3) if duration_delta is not None else None,\n        'durationDifferenceRatio': round(ratio, 4) if ratio is not None else None,\n        'exactSubdivision': exact_subdivision,\n        'sharedComponentTokens': shared_component_tokens,\n        'completeSapInterval': se is not None,\n        'assetMappingStatus': sap.get('mappingStatus', 'unavailable'),\n        'assetMappingValidated': sap.get('assetMappingValidated') is True,\n    }\n\n\ndef associate(mes_records, sap_records):\n    \"\"\"Return every MES decision plus the scored unique candidate pairs.\"\"\"\n    _validate_intervals(mes_records, sap_records)\n    candidates_by_mes = defaultdict(list)\n    candidates_by_sap = defaultdict(list)\n    for mes in mes_records:\n        for sap in sap_records:\n            if mes.get('machine') != sap.get('machine'):\n                continue\n            candidate = _candidate(mes, sap)\n            if candidate:\n                candidates_by_mes[mes['id']].append(candidate)\n                candidates_by_sap[sap['id']].append(candidate)\n\n    for candidates in candidates_by_mes.values():\n        candidates.sort(key=lambda row: (-row['score'], row['gapMinutes'] if row['gapMinutes'] is not None else 1e9,\n                                         abs(row['startDeltaMinutes']), row['sapId']))\n    for candidates in candidates_by_sap.values():\n        candidates.sort(key=lambda row: (-row['score'], row['gapMinutes'] if row['gapMinutes'] is not None else 1e9,\n                                         abs(row['startDeltaMinutes']), row['mesId']))\n\n    auto_pairs = set()\n    best_mes = {}\n    for mes_id, rows in candidates_by_mes.items():\n        best_mes[mes_id] = rows[0]\n    best_sap = {sap_id: rows[0] for sap_id, rows in candidates_by_sap.items()}\n    for mes_id, best in best_mes.items():\n        same_score = [row for row in candidates_by_mes[mes_id] if row['score'] == best['score']]\n        reverse = best_sap.get(best['sapId'])\n        sap_ties = [row for row in candidates_by_sap[best['sapId']] if row['score'] == reverse['score']] if reverse else []\n        next_mes = candidates_by_mes[mes_id][1]['score'] if len(candidates_by_mes[mes_id]) > 1 else -1\n        next_sap = candidates_by_sap[best['sapId']][1]['score'] if len(candidates_by_sap[best['sapId']]) > 1 else -1\n        strong_description = best['exactSubdivision'] or bool(best['sharedComponentTokens'])\n        if (reverse and reverse['mesId'] == mes_id and not same_score[1:] and not sap_ties[1:]\n                and best['score'] >= POLICY['autoMinimumScore']\n                and best['score'] - next_mes >= POLICY['minimumWinnerMargin']\n                and reverse['score'] - next_sap >= POLICY['minimumWinnerMargin']\n                and best['completeSapInterval'] and strong_description):\n            auto_pairs.add((mes_id, best['sapId']))\n\n    decisions = []\n    for mes in mes_records:\n        rows = candidates_by_mes.get(mes['id'], [])\n        auto = next((row for row in rows if (mes['id'], row['sapId']) in auto_pairs), None)\n        suggestion = next((row for row in rows if (mes['id'], row['sapId']) in auto_pairs), None)\n        if auto and auto['assetMappingValidated']:\n            status, selected, reason = 'auto_associated_unvalidated', auto['sapId'], 'strong_mutual_unique_rule'\n        elif suggestion:\n            status, selected, reason = 'suggested_unvalidated_mapping', None, 'asset_mapping_not_technically_validated'\n        elif not rows:\n            status, selected, reason = 'no_candidate', None, 'no_same_asset_interval_candidate_within_window'\n        elif any(not row['completeSapInterval'] for row in rows):\n            status, selected, reason = 'review_incomplete_interval', None, 'sap_interval_missing_end'\n        else:\n            top = rows[0]['score']\n            close = [row for row in rows if top - row['score'] < POLICY['minimumWinnerMargin']]\n            status = 'ambiguous' if len(close) > 1 else 'review_required'\n            selected = None\n            reason = 'competing_candidates' if status == 'ambiguous' else 'below_automatic_threshold_or_margin'\n        decisions.append({\n            'mesId': mes['id'], 'status': status, 'selectedSapId': selected,\n            'suggestedSapId': suggestion['sapId'] if suggestion and not selected else None,\n            'score': auto['score'] if auto else (rows[0]['score'] if rows else None),\n            'reason': reason,\n            'candidateCount': len(rows),\n            'candidates': [{k: row[k] for k in ('sapId', 'score', 'gapMinutes', 'overlapMinutes',\n                'startDeltaMinutes', 'endDeltaMinutes', 'durationDifferenceMinutes', 'exactSubdivision',\n                'sharedComponentTokens', 'completeSapInterval', 'assetMappingStatus', 'assetMappingValidated')} for row in rows[:5]],\n            'algorithmVersion': ALGORITHM_VERSION,\n        })\n    summary = {status: sum(row['status'] == status for row in decisions) for status in\n               ('auto_associated_unvalidated', 'suggested_unvalidated_mapping', 'ambiguous', 'review_required', 'review_incomplete_interval', 'no_candidate')}\n    return {'algorithmVersion': ALGORITHM_VERSION, 'policy': POLICY, 'decisions': decisions,\n            'automaticLinks': [\n                {'mesId': row['mesId'], 'sapId': row['selectedSapId'], 'score': row['score'],\n                 'status': row['status'], 'algorithmVersion': row['algorithmVersion']}\n                for row in decisions if row['selectedSapId']],\n            'automaticSuggestions': [\n                {'mesId': row['mesId'], 'sapId': row['suggestedSapId'], 'score': row['score'],\n                 'status': row['status'], 'algorithmVersion': row['algorithmVersion']}\n                for row in decisions if row['suggestedSapId']],\n            'candidates': [candidate for rows in candidates_by_mes.values() for candidate in rows],\n            'summary': summary}\n",
 "05_subprodutos/02_conciliacao/conciliacao_evidencias.py": "\"\"\"Método candidato `sp2-evidence-v1`: conciliação MES × SAP por equipamento, tempo e evidências.\r\n\r\nRoda em paralelo à v3 (`associacao_automatica.py`), que não é alterada. Todas as regras e limiares\r\nabaixo são propostas, não valores validados; classes (forte/moderada/fraca) são ordinais, não\r\nprobabilidades, e nenhuma decisão confirma correspondência física nem valida o cadastro do ativo.\r\nFunções puras sobre registros no formato do histórico (`mes`/`sap`).\r\n\"\"\"\r\nfrom bisect import bisect_left, bisect_right\r\nfrom collections import defaultdict\r\nfrom datetime import timedelta\r\nimport hashlib\r\nimport json\r\n\r\ntry:  # execução como pacote ou com o diretório no sys.path\r\n    from associacao_automatica import normalize, _dt, _validate_intervals\r\nexcept ImportError:  # pragma: no cover\r\n    from .associacao_automatica import normalize, _dt, _validate_intervals\r\n\r\nRULES_VERSION = 'sp2-evidence-v1'\r\nCLASS_RANK = {'forte': 0, 'moderada': 1, 'fraca': 2}\r\nSTRONG_TEMPORAL = ('sobreposicao', 'contencao', 'contato')\r\nRULES = {\r\n    'rulesVersion': RULES_VERSION,\r\n    'status': 'proposta; limiares não calibrados nem validados por humanos',\r\n    'candidateWindowMinutes': 60,\r\n    'candidateRule': 'mesma máquina; SAP com fim: gap <= janela; SAP sem fim: -janela <= início SAP - início MES <= duração MES + janela (idêntico à v3)',\r\n    'shortProximityMinutes': 15,\r\n    'sapLongWindowMinutes': 24 * 60,\r\n    'fragmentationMaxGapMinutes': 1,\r\n    'nmMaxMembers': 12,\r\n    'durationComparison': {'method': 'duration-v1', 'absoluteThresholdMinutes': 15, 'relativeThreshold': 0.25,\r\n                           'rule': 'relevante quando |diferença| >= 15 min E |diferença| / max(durações) >= 0,25; triagem proposta, não validada.',\r\n                           'bothZero': 'razão 0 por convenção explícita; não implica horas efetivas',\r\n                           'unavailable': 'sem duração de algum lado: diferença, razão e relevância ficam nulas'},\r\n    'classes': {\r\n        'forte': 'temporal em {sobreposição, contenção, contato} E (subdivisão igual OU componente citado no texto SAP)',\r\n        'moderada': 'temporal forte sem conteúdo positivo; OU proximidade (curta ou até a janela) com conteúdo positivo',\r\n        'fraca': 'demais casos, incluindo SAP sem fim e proximidade sem conteúdo positivo',\r\n        'nota': 'ordinais; ausência de informação não é contradição nem evidência positiva; incompatível só existiria com mapa técnico (hoje vazio)',\r\n    },\r\n    'decisions': {\r\n        'sem_candidato': 'nenhum registro da outra fonte, mesma máquina, dentro da regra de candidatos/janela',\r\n        'correspondencia_provavel': 'um único candidato forte e ele é recíproco (a contraparte também tem um único forte); pela regra, cadastro inferido, revisão pendente',\r\n        'disputa': 'dois ou mais fortes neste lado ou na contraparte; sem desempate automático',\r\n        'evidencia_insuficiente': 'há candidatos, mas nenhum forte',\r\n    },\r\n    'subdivisionRule': 'subdivisões diferentes: compatibilidade desconhecida (não há mapa técnico que sustente incompatibilidade)',\r\n    'componentRule': 'tokens do componente MES com >= 3 caracteres presentes no texto SAP (mesma regra da v3)',\r\n    'blockRule': 'componentes conexos do grafo de candidatos por máquina; unidade de investigação, não evento confirmado',\r\n    'hypothesisRule': 'hipóteses 1:1 (pares fortes/moderados), N:1, 1:N e N:M (subgrafo forte/moderado; até 12 membros) são propostas concorrentes; nunca une fragmentos nem força 1:1',\r\n    'flags': {\r\n        'duplicata_exata': 'todos os campos originais (raw) idênticos na mesma fonte; não prova duplicidade física',\r\n        'possivel_duplicidade': 'mesma fonte, máquina, subdivisão e componente (MES) ou texto (SAP) normalizados e não vazios, com intervalos sobrepostos; texto igual ou simultaneidade isolados não bastam',\r\n        'possivel_fragmentacao': 'MES da mesma máquina e subdivisão, componente igual ou ausente em algum lado, sem sobreposição e com intervalo entre 0 e 1 min; triagem, não continuidade confirmada',\r\n    },\r\n    'labels': {\r\n        'decisoes': {'sem_candidato': 'Sem candidato', 'correspondencia_provavel': 'Correspondência provável',\r\n                     'disputa': 'Disputa', 'evidencia_insuficiente': 'Evidência insuficiente'},\r\n        'motivos': {\r\n            'forte_unico_reciproco': 'Um único candidato forte, recíproco',\r\n            'varios_fortes_neste_lado': 'Dois ou mais candidatos fortes neste registro',\r\n            'contraparte_com_varios_fortes': 'O candidato forte tem outros fortes concorrentes',\r\n            'sem_forte': 'Há candidatos, mas nenhum forte',\r\n            'melhor_classe_moderada': 'Melhor candidato é moderado',\r\n            'melhor_classe_fraca': 'Melhor candidato é fraco',\r\n            'candidato_sap_sem_fim': 'Há candidato SAP sem fim registrado',\r\n            'sem_registro_na_outra_fonte': 'Nenhum registro da outra fonte na janela',\r\n            'intervalo_sap_sem_fim': 'Janela SAP sem fim registrado',\r\n            'cadastro_inferido': 'Cadastro do ativo SAP inferido, não validado',\r\n            'revisao_pendente': 'Revisão humana pendente',\r\n        },\r\n        'confianca': {'evidencia_insuficiente': 'Evidência insuficiente', 'disputa': 'Disputa',\r\n                      'intervalo_incompleto': 'Intervalo incompleto', 'cadastro_pendente': 'Cadastro pendente'},\r\n        'saidas': {'provavel': 'Correspondência provável', 'mes_sem_candidato': 'MES sem candidato',\r\n                   'sap_sem_candidato': 'SAP sem candidato', 'possivel_duplicidade': 'Possíveis duplicidades',\r\n                   'divergencia_duracao': 'Diferença relevante de duração', 'confianca_insuficiente': 'Confiança insuficiente'},\r\n    },\r\n}\r\nOUTPUT_KEYS = tuple(RULES['labels']['saidas'])\r\nCONFIDENCE_KEYS = tuple(RULES['labels']['confianca'])\r\n\r\n\r\ndef canonical(value):\r\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))\r\n\r\n\r\ndef content_hash(value):\r\n    return hashlib.sha256(canonical(value).encode('utf-8')).hexdigest()\r\n\r\n\r\ndef duration_divergence(mes_duration, sap_duration):\r\n    \"\"\"Mesma semântica de `gerar_historico_mvp.duration_divergence` (duration-v1), com comparabilidade explícita.\"\"\"\r\n    cfg = RULES['durationComparison']\r\n    if mes_duration is None or sap_duration is None:\r\n        return {'comparavel': False, 'diferencaMin': None, 'absolutaMin': None, 'razao': None, 'relevante': None,\r\n                'metodo': cfg['method']}\r\n    difference = round(float(sap_duration) - float(mes_duration), 6)\r\n    absolute = round(abs(difference), 6)\r\n    denominator = max(float(mes_duration), float(sap_duration))\r\n    exact_ratio = abs(difference) / denominator if denominator else 0.0\r\n    relevant = absolute >= cfg['absoluteThresholdMinutes'] and exact_ratio >= cfg['relativeThreshold']\r\n    return {'comparavel': True, 'diferencaMin': difference, 'absolutaMin': absolute, 'razao': round(exact_ratio, 6),\r\n            'relevante': relevant, 'metodo': cfg['method']}\r\n\r\n\r\n# ---------------------------------------------------------------- registros e candidatos\r\n\r\ndef _prepare(rows, system):\r\n    prepared = []\r\n    for index, row in enumerate(rows):\r\n        start, end = _dt(row.get('start')), _dt(row.get('end'))\r\n        duration = row.get('durationMinutes')\r\n        if duration is None and start is not None and end is not None:\r\n            duration = round((end - start).total_seconds() / 60, 6)\r\n        prepared.append({\r\n            'id': row['id'], 'system': system, 'index': index, 'machine': row.get('machine'), 'start': start, 'end': end,\r\n            'duration': duration, 'subdivision': normalize(row.get('subdivision')),\r\n            'component': normalize(row.get('component')), 'text': normalize(row.get('text')),\r\n            'tokens': set(normalize(row.get('text')).split()), 'row': row,\r\n        })\r\n    return prepared\r\n\r\n\r\ndef _geometry(mes, sap, window):\r\n    \"\"\"Predicado de candidatos idêntico ao da v3 (`associacao_automatica._candidate`), em minutos float.\"\"\"\r\n    duration_m = (mes['end'] - mes['start']).total_seconds() / 60\r\n    start_delta = (sap['start'] - mes['start']).total_seconds() / 60\r\n    if sap['end'] is not None:\r\n        gap = max(0.0, (sap['start'] - mes['end']).total_seconds() / 60, (mes['start'] - sap['end']).total_seconds() / 60)\r\n        if gap > window:\r\n            return None\r\n        overlap = max(0.0, (min(mes['end'], sap['end']) - max(mes['start'], sap['start'])).total_seconds() / 60)\r\n    else:\r\n        if start_delta < -window or start_delta > duration_m + window:\r\n            return None\r\n        gap, overlap = None, None\r\n    end_delta = (sap['end'] - mes['end']).total_seconds() / 60 if sap['end'] is not None else None\r\n    return {'gap': gap, 'overlap': overlap, 'startDelta': start_delta, 'endDelta': end_delta}\r\n\r\n\r\ndef brute_force_pairs(mes_records, sap_records, window=60):\r\n    \"\"\"Referência O(n·m) para provar a equivalência com o índice (e com a v3).\"\"\"\r\n    mes, sap = _prepare(mes_records, 'MES'), _prepare(sap_records, 'SAP')\r\n    return [(m['id'], s['id']) for m in mes for s in sap\r\n            if m['machine'] == s['machine'] and _geometry(m, s, window) is not None]\r\n\r\n\r\ndef _indexed_pairs(mes, sap, window):\r\n    by_machine = defaultdict(list)\r\n    for item in sap:\r\n        by_machine[item['machine']].append(item)\r\n    index = {}\r\n    for machine, items in by_machine.items():\r\n        items.sort(key=lambda item: (item['start'], item['index']))\r\n        longest = max([(i['end'] - i['start']) for i in items if i['end'] is not None] or [timedelta(0)])\r\n        index[machine] = (items, [item['start'] for item in items], longest)\r\n    slack, span = timedelta(seconds=1), timedelta(minutes=window)\r\n    pairs = []\r\n    for m in mes:\r\n        if m['machine'] not in index:\r\n            continue\r\n        items, starts, longest = index[m['machine']]\r\n        low = bisect_left(starts, m['start'] - span - longest - slack)\r\n        high = bisect_right(starts, m['end'] + span + slack)\r\n        for s in sorted(items[low:high], key=lambda item: item['index']):\r\n            geometry = _geometry(m, s, window)\r\n            if geometry is not None:\r\n                pairs.append((m, s, geometry))\r\n    return pairs\r\n\r\n\r\ndef indexed_pairs(mes_records, sap_records, window=60):\r\n    \"\"\"Pares (mesId, sapId) por índice máquina/tempo; valida intervalos como a v3.\"\"\"\r\n    _validate_intervals(mes_records, sap_records)\r\n    return [(m['id'], s['id']) for m, s, _ in _indexed_pairs(_prepare(mes_records, 'MES'), _prepare(sap_records, 'SAP'), window)]\r\n\r\n\r\n# ---------------------------------------------------------------- sinais e classes\r\n\r\ndef _temporal(m, s, geometry):\r\n    if geometry['gap'] is None:\r\n        return 'sap_sem_fim'\r\n    contained = (s['start'] <= m['start'] and s['end'] >= m['end']) or (m['start'] <= s['start'] and m['end'] >= s['end'])\r\n    if contained:\r\n        return 'contencao'\r\n    if geometry['overlap'] > 0:\r\n        return 'sobreposicao'\r\n    if geometry['gap'] == 0:\r\n        return 'contato'\r\n    return 'proximidade_curta' if geometry['gap'] <= RULES['shortProximityMinutes'] else 'proximidade'\r\n\r\n\r\ndef _signals(m, s, geometry, window, v3_scores):\r\n    temporal = _temporal(m, s, geometry)\r\n    if not m['subdivision'] or not s['subdivision']:\r\n        subdivision = 'ausente'\r\n    elif m['subdivision'] == s['subdivision']:\r\n        subdivision = 'igual'\r\n    else:\r\n        subdivision = 'diferente_compatibilidade_desconhecida'\r\n    tokens = sorted({t for t in m['component'].split() if len(t) >= 3} & s['tokens'])\r\n    if tokens:\r\n        component = 'citado'\r\n    elif not m['component'] or not s['text']:\r\n        component = 'ausente'\r\n    else:\r\n        component = 'nao_citado'\r\n    positive = subdivision == 'igual' or component == 'citado'\r\n    if temporal in STRONG_TEMPORAL:\r\n        klass = 'forte' if positive else 'moderada'\r\n    elif temporal in ('proximidade_curta', 'proximidade') and positive:\r\n        klass = 'moderada'\r\n    else:\r\n        klass = 'fraca'\r\n    row = s['row']\r\n    validated = row.get('assetMappingValidated') is True\r\n    status = row.get('mappingStatus', 'unavailable')\r\n    cadastro = 'validado' if validated else 'inferido' if 'inferido' in str(status) else 'indisponivel'\r\n    long_window = s['duration'] is not None and s['duration'] > RULES['sapLongWindowMinutes']\r\n    return {\r\n        'id': f\"{m['id']}|{s['id']}\", 'mesId': m['id'], 'sapId': s['id'], 'maquina': m['machine'], 'janelaMin': window,\r\n        'temporal': temporal, 'gapMin': _r(geometry['gap']), 'sobreposicaoMin': _r(geometry['overlap']),\r\n        'deltaInicioMin': _r(geometry['startDelta']), 'deltaFimMin': _r(geometry['endDelta']),\r\n        'subdivisao': subdivision, 'componente': component, 'tokensComponente': tokens,\r\n        'cadastro': cadastro, 'mappingStatus': status, 'assetMappingValidated': validated,\r\n        'duracao': duration_divergence(m['duration'], s['duration']),\r\n        'sapSemFim': s['end'] is None, 'janelaSapLonga': long_window,\r\n        'classe': klass, 'scoreV3': (v3_scores or {}).get((m['id'], s['id'])),\r\n    }\r\n\r\n\r\ndef _r(value):\r\n    return None if value is None else round(value, 3)\r\n\r\n\r\n# ---------------------------------------------------------------- união, decisões, blocos, hipóteses\r\n\r\ndef _iso(value):\r\n    return value.isoformat()\r\n\r\n\r\ndef union_intervals(intervals):\r\n    \"\"\"União de (início, fim) datetimes; devolve intervalos mesclados, duração somada e lacunas.\"\"\"\r\n    merged = []\r\n    for start, end in sorted(intervals):\r\n        if merged and start <= merged[-1][1]:\r\n            if end > merged[-1][1]:\r\n                merged[-1][1] = end\r\n        else:\r\n            merged.append([start, end])\r\n    total = sum((end - start).total_seconds() / 60 for start, end in merged)\r\n    gaps = [{'inicio': _iso(a[1]), 'fim': _iso(b[0]), 'min': round((b[0] - a[1]).total_seconds() / 60, 3)}\r\n            for a, b in zip(merged, merged[1:])]\r\n    return {'intervalos': [[_iso(a), _iso(b)] for a, b in merged], 'duracaoMin': round(total, 3), 'lacunas': gaps}\r\n\r\n\r\ndef _side_union(records):\r\n    complete = [(r['start'], r['end']) for r in records if r['end'] is not None]\r\n    result = union_intervals(complete)\r\n    result['registros'] = len(records)\r\n    result['semFim'] = [r['id'] for r in records if r['end'] is None]\r\n    result['incompleto'] = bool(result['semFim'])\r\n    return result\r\n\r\n\r\ndef _hypothesis(kind, mes_members, sap_members, block_id, pair_rows):\r\n    uniao = {'MES': _side_union(mes_members), 'SAP': _side_union(sap_members)}\r\n    incomplete = uniao['SAP']['incompleto'] or uniao['MES']['incompleto'] or not mes_members or not sap_members\r\n    if incomplete:\r\n        comparison = duration_divergence(None, None)\r\n        comparison['motivo'] = 'nao_calculavel_intervalo_incompleto'\r\n    else:\r\n        comparison = duration_divergence(uniao['MES']['duracaoMin'], uniao['SAP']['duracaoMin'])\r\n        if uniao['MES']['duracaoMin'] == 0 and uniao['SAP']['duracaoMin'] == 0:\r\n            comparison['motivo'] = 'ambas_duracoes_zero_convencao_razao_zero'\r\n    return {\r\n        'tipo': kind, 'estado': 'proposta', 'blocoId': block_id,\r\n        'mesIds': [r['id'] for r in mes_members], 'sapIds': [r['id'] for r in sap_members],\r\n        'pares': sorted(pair_rows), 'uniao': uniao,\r\n        'janelaSapLonga': [r['id'] for r in sap_members if r['duration'] is not None and r['duration'] > RULES['sapLongWindowMinutes']],\r\n        'incompleto': incomplete, 'duracao': comparison,\r\n        'nota': 'Hipótese concorrente para investigação; não confirma evento físico nem une fragmentos. Duração = união dos intervalos registrados por fonte.',\r\n    }\r\n\r\n\r\ndef _components(nodes, edges):\r\n    parent = {n: n for n in nodes}\r\n\r\n    def find(x):\r\n        while parent[x] != x:\r\n            parent[x] = parent[parent[x]]\r\n            x = parent[x]\r\n        return x\r\n    for a, b in edges:\r\n        ra, rb = find(a), find(b)\r\n        if ra != rb:\r\n            parent[max(ra, rb)] = min(ra, rb)\r\n    groups = defaultdict(list)\r\n    for n in nodes:\r\n        groups[find(n)].append(n)\r\n    return list(groups.values())\r\n\r\n\r\ndef _flag_overlap(a, b):\r\n    if a['end'] is None or b['end'] is None:\r\n        return False\r\n    return min(a['end'], b['end']) > max(a['start'], b['start'])\r\n\r\n\r\ndef _flags(mes, sap):\r\n    flags = []\r\n    for system, records in (('MES', mes), ('SAP', sap)):\r\n        # duplicata exata: raw idêntico na mesma fonte\r\n        by_raw = defaultdict(list)\r\n        for r in records:\r\n            by_raw[canonical(r['row'].get('raw', {}))].append(r)\r\n        for group in by_raw.values():\r\n            if len(group) > 1 and group[0]['row'].get('raw'):\r\n                flags.append(_flag('duplicata_exata', system, group, {'raw': 'idêntico'}))\r\n        exact_ids = {r['id'] for g in by_raw.values() if len(g) > 1 and g[0]['row'].get('raw') for r in g}\r\n        # possível duplicidade: chave normalizada + sobreposição\r\n        key_of = (lambda r: (r['machine'], r['subdivision'], r['component'])) if system == 'MES' else (lambda r: (r['machine'], r['subdivision'], r['text']))\r\n        groups = defaultdict(list)\r\n        for r in records:\r\n            key = key_of(r)\r\n            if key[1] and key[2] and r['end'] is not None:\r\n                groups[key].append(r)\r\n        for key, group in groups.items():\r\n            group.sort(key=lambda r: (r['start'], r['index']))\r\n            for i, a in enumerate(group):\r\n                for b in group[i + 1:]:\r\n                    if b['start'] >= a['end']:\r\n                        break\r\n                    if _flag_overlap(a, b) and not (a['id'] in exact_ids and b['id'] in exact_ids\r\n                                                    and canonical(a['row'].get('raw')) == canonical(b['row'].get('raw'))):\r\n                        fields = {'maquina': key[0], 'subdivisao': key[1], ('componente' if system == 'MES' else 'texto'): key[2],\r\n                                  'sobreposicaoMin': round((min(a['end'], b['end']) - max(a['start'], b['start'])).total_seconds() / 60, 3)}\r\n                        flags.append(_flag('possivel_duplicidade', system, [a, b], fields))\r\n    # possível fragmentação: MES adjacentes (gap 0..1 min), mesma subdivisão, componente igual/ausente\r\n    by_machine = defaultdict(list)\r\n    for r in mes:\r\n        by_machine[r['machine']].append(r)\r\n    limit = RULES['fragmentationMaxGapMinutes']\r\n    for machine, group in by_machine.items():\r\n        group.sort(key=lambda r: (r['start'], r['index']))\r\n        for i, a in enumerate(group):\r\n            for b in group[i + 1:]:\r\n                gap = (b['start'] - a['end']).total_seconds() / 60\r\n                if gap > limit:\r\n                    break\r\n                if gap < 0 or not a['subdivision'] or a['subdivision'] != b['subdivision']:\r\n                    continue\r\n                if a['component'] and b['component'] and a['component'] != b['component']:\r\n                    continue\r\n                flags.append(_flag('possivel_fragmentacao', 'MES', [a, b],\r\n                                   {'maquina': machine, 'subdivisao': a['subdivision'], 'componenteA': a['component'] or None,\r\n                                    'componenteB': b['component'] or None, 'gapMin': round(gap, 3)}))\r\n    for index, flag in enumerate(sorted(flags, key=lambda f: (f['tipo'], f['sistema'], f['registros'])), 1):\r\n        flag['id'] = f'FLG-{index:05d}'\r\n    return sorted(flags, key=lambda f: f['id'])\r\n\r\n\r\ndef _flag(kind, system, records, fields):\r\n    return {'id': None, 'tipo': kind, 'sistema': system, 'registros': [r['id'] for r in records],\r\n            'regra': RULES['flags'][kind], 'campos': fields, 'estado': 'proposta',\r\n            'nota': 'Sinalização para revisão; nenhum registro é apagado ou unido.'}\r\n\r\n\r\n# ---------------------------------------------------------------- análise principal\r\n\r\ndef analyze(mes_records, sap_records, window=60, v3_scores=None):\r\n    \"\"\"Executa o método candidato e devolve decisões, pares/sinais, blocos, hipóteses, flags e saídas.\"\"\"\r\n    _validate_intervals(mes_records, sap_records)\r\n    ids = [r['id'] for r in mes_records] + [r['id'] for r in sap_records]\r\n    if len(set(ids)) != len(ids):\r\n        raise ValueError('IDs duplicados entre os registros MES/SAP.')\r\n    mes, sap = _prepare(mes_records, 'MES'), _prepare(sap_records, 'SAP')\r\n    by_id = {r['id']: r for r in mes + sap}\r\n    pairs = [_signals(m, s, g, window, v3_scores) for m, s, g in _indexed_pairs(mes, sap, window)]\r\n    pair_by_id = {p['id']: p for p in pairs}\r\n    by_record = defaultdict(list)\r\n    for p in pairs:\r\n        by_record[p['mesId']].append(p)\r\n        by_record[p['sapId']].append(p)\r\n\r\n    def order(p):\r\n        return (CLASS_RANK[p['classe']], p['gapMin'] if p['gapMin'] is not None else 1e9, abs(p['deltaInicioMin']), p['id'])\r\n\r\n    # blocos\r\n    graph_nodes = [r['id'] for r in mes + sap if by_record.get(r['id'])]\r\n    blocks_raw = _components(graph_nodes, [(p['mesId'], p['sapId']) for p in pairs])\r\n    blocks_raw.sort(key=lambda g: (by_id[g[0]]['machine'] or '', min(by_id[i]['start'] for i in g), min(g)))\r\n    block_of, blocks, counter = {}, [], defaultdict(int)\r\n    for members in blocks_raw:\r\n        machine = by_id[members[0]]['machine']\r\n        counter[machine] += 1\r\n        block_id = f'BLK-{machine}-{counter[machine]:04d}'\r\n        members = sorted(members)\r\n        for member in members:\r\n            block_of[member] = block_id\r\n        blocks.append({'id': block_id, 'maquina': machine, 'rotulo': 'unidade de investigação; não é evento confirmado',\r\n                       'mesIds': [i for i in members if by_id[i]['system'] == 'MES'],\r\n                       'sapIds': [i for i in members if by_id[i]['system'] == 'SAP'],\r\n                       'nPares': 0,\r\n                       'hipoteseIds': []})\r\n    blocks_by_id = {b['id']: b for b in blocks}\r\n    for p in pairs:\r\n        blocks_by_id[block_of[p['mesId']]]['nPares'] += 1\r\n\r\n    # decisões\r\n    def strong_of(record_id):\r\n        return [p for p in by_record.get(record_id, []) if p['classe'] == 'forte']\r\n\r\n    decisions = {}\r\n    for r in mes + sap:\r\n        rows = sorted(by_record.get(r['id'], []), key=order)\r\n        own_system = r['system']\r\n        counterpart = lambda p: p['sapId'] if own_system == 'MES' else p['mesId']\r\n        strong = [p for p in rows if p['classe'] == 'forte']\r\n        motives, counter_id = [], None\r\n        if not rows:\r\n            decision = 'sem_candidato'\r\n            motives.append('sem_registro_na_outra_fonte')\r\n        elif len(strong) >= 2:\r\n            decision = 'disputa'\r\n            motives.append('varios_fortes_neste_lado')\r\n        elif len(strong) == 1:\r\n            counter_id = counterpart(strong[0])\r\n            if len(strong_of(counter_id)) >= 2:\r\n                decision = 'disputa'\r\n                motives.append('contraparte_com_varios_fortes')\r\n            else:\r\n                decision = 'correspondencia_provavel'\r\n                motives.append('forte_unico_reciproco')\r\n        else:\r\n            decision = 'evidencia_insuficiente'\r\n            motives += ['sem_forte', 'melhor_classe_moderada' if rows[0]['classe'] == 'moderada' else 'melhor_classe_fraca']\r\n        sap_open = own_system == 'SAP' and r['end'] is None\r\n        alt_open = any(p['sapSemFim'] for p in rows)\r\n        if sap_open:\r\n            motives.append('intervalo_sap_sem_fim')\r\n        if own_system == 'MES' and alt_open:\r\n            motives.append('candidato_sap_sem_fim')\r\n        if own_system == 'SAP':\r\n            cad = 'validado' if r['row'].get('assetMappingValidated') is True else 'inferido' if 'inferido' in str(r['row'].get('mappingStatus')) else 'indisponivel'\r\n        else:\r\n            cad = 'nao_aplicavel' if not rows else ('validado' if all(p['cadastro'] == 'validado' for p in rows) else 'inferido'\r\n                                                  if any(p['cadastro'] == 'inferido' for p in rows) else 'indisponivel')\r\n        if cad == 'inferido':\r\n            motives.append('cadastro_inferido')\r\n        motives.append('revisao_pendente')\r\n        confidence = []\r\n        if rows:\r\n            if decision == 'evidencia_insuficiente':\r\n                confidence.append('evidencia_insuficiente')\r\n            if decision == 'disputa':\r\n                confidence.append('disputa')\r\n            if cad in ('inferido', 'indisponivel'):\r\n                confidence.append('cadastro_pendente')\r\n        if sap_open or alt_open:\r\n            confidence.append('intervalo_incompleto')\r\n        decisions[r['id']] = {\r\n            'id': r['id'], 'sistema': r['system'], 'maquina': r['machine'], 'decisao': decision,\r\n            'classeMaxima': rows[0]['classe'] if rows else None, 'contraparteId': counter_id,\r\n            'nCandidatos': len(rows), 'nForte': len(strong),\r\n            'nModerada': sum(p['classe'] == 'moderada' for p in rows), 'nFraca': sum(p['classe'] == 'fraca' for p in rows),\r\n            'alternativas': [{'id': counterpart(p), 'classe': p['classe'], 'par': p['id']} for p in rows],\r\n            'motivos': motives, 'motivosConfianca': confidence, 'cadastro': cad,\r\n            'intervaloIncompleto': sap_open, 'alternativaIncompleta': alt_open,\r\n            'revisao': 'pendente', 'blocoId': block_of.get(r['id']), 'saidas': [],\r\n        }\r\n\r\n    # hipóteses por bloco (subgrafo forte/moderado)\r\n    hypotheses = []\r\n\r\n    def add(kind, mes_ids, sap_ids, block_id, pair_ids):\r\n        h = _hypothesis(kind, [by_id[i] for i in mes_ids], [by_id[i] for i in sap_ids], block_id, pair_ids)\r\n        h['id'] = f'HYP-{len(hypotheses) + 1:05d}'\r\n        hypotheses.append(h)\r\n        blocks_by_id[block_id]['hipoteseIds'].append(h['id'])\r\n        return h\r\n\r\n    fm_by_sap, fm_by_mes = defaultdict(list), defaultdict(list)\r\n    for p in pairs:\r\n        if p['classe'] in ('forte', 'moderada'):\r\n            fm_by_sap[p['sapId']].append(p)\r\n            fm_by_mes[p['mesId']].append(p)\r\n    for block in blocks:\r\n        block_pairs = [p for p in pairs if block_of[p['mesId']] == block['id'] and p['classe'] in ('forte', 'moderada')]\r\n        for p in block_pairs:\r\n            add('1:1', [p['mesId']], [p['sapId']], block['id'], [p['id']])\r\n        for sap_id in block['sapIds']:\r\n            rows = fm_by_sap.get(sap_id, [])\r\n            if len(rows) >= 2:\r\n                add('N:1', [p['mesId'] for p in rows], [sap_id], block['id'], [p['id'] for p in rows])\r\n        for mes_id in block['mesIds']:\r\n            rows = fm_by_mes.get(mes_id, [])\r\n            if len(rows) >= 2:\r\n                add('1:N', [mes_id], [p['sapId'] for p in rows], block['id'], [p['id'] for p in rows])\r\n        if block_pairs:\r\n            nodes = sorted({p['mesId'] for p in block_pairs} | {p['sapId'] for p in block_pairs})\r\n            for component in _components(nodes, [(p['mesId'], p['sapId']) for p in block_pairs]):\r\n                mes_ids = sorted(i for i in component if by_id[i]['system'] == 'MES')\r\n                sap_ids = sorted(i for i in component if by_id[i]['system'] == 'SAP')\r\n                if len(mes_ids) >= 2 and len(sap_ids) >= 2:\r\n                    members = {*mes_ids, *sap_ids}\r\n                    rows = [p for p in block_pairs if p['mesId'] in members]\r\n                    h = add('N:M', mes_ids, sap_ids, block['id'], [p['id'] for p in rows])\r\n                    if len(members) > RULES['nmMaxMembers']:\r\n                        h['estado'] = 'nm_nao_enumerado'\r\n                        h['nota'] += f\" Mais de {RULES['nmMaxMembers']} membros: alternativas internas não enumeradas.\"\r\n    member_hyp = defaultdict(list)\r\n    for h in hypotheses:\r\n        for i in h['mesIds'] + h['sapIds']:\r\n            member_hyp[i].append(h['id'])\r\n    for h in hypotheses:\r\n        others = {o for i in h['mesIds'] + h['sapIds'] for o in member_hyp[i]} - {h['id']}\r\n        h['alternativas'] = sorted(others)\r\n    for r_id, ids_ in member_hyp.items():\r\n        decisions[r_id]['hipoteseIds'] = ids_\r\n    for d in decisions.values():\r\n        d.setdefault('hipoteseIds', [])\r\n\r\n    # flags\r\n    flags = _flags(mes, sap)\r\n    flagged = defaultdict(list)\r\n    for f in flags:\r\n        for r_id in f['registros']:\r\n            flagged[r_id].append(f['id'])\r\n    # saídas\r\n    flag_type = {f['id']: f['tipo'] for f in flags}\r\n    divergent = defaultdict(bool)\r\n    for p in pairs:\r\n        if p['classe'] in ('forte', 'moderada') and p['duracao']['relevante']:\r\n            divergent[p['mesId']] = divergent[p['sapId']] = True\r\n    for h in hypotheses:\r\n        if h['duracao'].get('relevante'):\r\n            for i in h['mesIds'] + h['sapIds']:\r\n                divergent[i] = True\r\n    for d in decisions.values():\r\n        outputs = d['saidas']\r\n        if d['decisao'] == 'correspondencia_provavel':\r\n            outputs.append('provavel')\r\n        if d['decisao'] == 'sem_candidato':\r\n            outputs.append('mes_sem_candidato' if d['sistema'] == 'MES' else 'sap_sem_candidato')\r\n        if any(flag_type[f] in ('duplicata_exata', 'possivel_duplicidade') for f in flagged.get(d['id'], [])):\r\n            outputs.append('possivel_duplicidade')\r\n        if divergent[d['id']]:\r\n            outputs.append('divergencia_duracao')\r\n        if d['motivosConfianca']:\r\n            outputs.append('confianca_insuficiente')\r\n        d['flagIds'] = flagged.get(d['id'], [])\r\n        outputs.sort(key=OUTPUT_KEYS.index)\r\n\r\n    decision_list = [decisions[r['id']] for r in mes] + [decisions[r['id']] for r in sap]\r\n    summary = _summary(decision_list, pairs, hypotheses, flags, blocks)\r\n    summary['sapJanelaLonga'] = sum(1 for r in sap if r['duration'] is not None and r['duration'] > RULES['sapLongWindowMinutes'])\r\n    return {'rulesVersion': RULES_VERSION, 'window': window, 'decisions': decision_list, 'pairs': pairs,\r\n            'blocks': blocks, 'hypotheses': hypotheses, 'flags': flags, 'summary': summary}\r\n\r\n\r\ndef _count(values):\r\n    result = defaultdict(int)\r\n    for v in values:\r\n        result[v] += 1\r\n    return dict(sorted(result.items()))\r\n\r\n\r\ndef _summary(decisions, pairs, hypotheses, flags, blocks):\r\n    summary = {\r\n        'registros': {'MES': sum(d['sistema'] == 'MES' for d in decisions), 'SAP': sum(d['sistema'] == 'SAP' for d in decisions)},\r\n        'decisoes': {s: _count(d['decisao'] for d in decisions if d['sistema'] == s) for s in ('MES', 'SAP')},\r\n        'pares': len(pairs), 'classesPares': _count(p['classe'] for p in pairs),\r\n        'temporalPares': _count(p['temporal'] for p in pairs), 'blocos': len(blocks),\r\n        'hipoteses': _count(h['tipo'] for h in hypotheses), 'flags': _count(f['tipo'] for f in flags),\r\n        'saidas': {k: sum(k in d['saidas'] for d in decisions) for k in OUTPUT_KEYS},\r\n        'saidasPorSistema': {s: {k: sum(k in d['saidas'] for d in decisions if d['sistema'] == s) for k in OUTPUT_KEYS} for s in ('MES', 'SAP')},\r\n        'confiancaPorMotivo': {k: sum(k in d['motivosConfianca'] for d in decisions) for k in CONFIDENCE_KEYS},\r\n        'cadastro': _count(d['cadastro'] for d in decisions),\r\n    }\r\n    return summary\r\n\r\n\r\ndef analyze_history(history, window=60):\r\n    \"\"\"Atalho para o formato do histórico: usa os scores v3 embutidos, quando a janela coincide.\"\"\"\r\n    scores = {}\r\n    if window == 60 and isinstance(history.get('association'), dict):\r\n        scores = {(c['mesId'], c['sapId']): c['score'] for c in history['association'].get('candidates', [])}\r\n    return analyze(history['mes'], history['sap'], window, scores)\r\n",
 "05_subprodutos/02_conciliacao/gerar_snapshot_evidencias.py": "\"\"\"Gera snapshot JSON, base SQLite e histórico aditivo do método candidato `sp2-evidence-v1`.\r\n\r\nEntrada: histórico já publicado (não regenera via `gerar_historico_mvp`). Saídas com nomes novos; recusa sobrescrever.\r\nO hash de conteúdo exclui `generatedAt`. Nenhuma revisão humana é simulada: `reviews` e `ev_review` nascem vazios.\r\n\"\"\"\r\nimport argparse\r\nimport hashlib\r\nimport json\r\nimport sqlite3\r\nfrom datetime import datetime, timezone\r\nfrom pathlib import Path\r\nfrom uuid import uuid4\r\n\r\ntry:\r\n    from associacao_automatica import associate\r\n    from criar_base_consolidada import SCHEMA, canonical\r\n    import conciliacao_evidencias as ce\r\nexcept ImportError:  # pragma: no cover\r\n    from .associacao_automatica import associate\r\n    from .criar_base_consolidada import SCHEMA, canonical\r\n    from . import conciliacao_evidencias as ce\r\n\r\nEV_SCHEMA = '''\r\nCREATE TABLE ev_rule_version (\r\n  dataset_id TEXT NOT NULL,\r\n  rules_version TEXT NOT NULL,\r\n  window_minutes INTEGER NOT NULL,\r\n  input_sha256 TEXT NOT NULL CHECK(length(input_sha256)=64),\r\n  content_sha256 TEXT NOT NULL CHECK(length(content_sha256)=64),\r\n  rules_json TEXT NOT NULL CHECK(json_valid(rules_json)),\r\n  PRIMARY KEY(dataset_id, rules_version),\r\n  FOREIGN KEY(dataset_id) REFERENCES dataset_version(dataset_id)\r\n);\r\nCREATE TABLE ev_pair_signal (\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL,\r\n  mes_id TEXT NOT NULL, mes_system TEXT NOT NULL DEFAULT 'MES' CHECK(mes_system='MES'),\r\n  sap_id TEXT NOT NULL, sap_system TEXT NOT NULL DEFAULT 'SAP' CHECK(sap_system='SAP'),\r\n  classe TEXT NOT NULL CHECK(classe IN ('forte','moderada','fraca')),\r\n  temporal TEXT NOT NULL, subdivisao TEXT NOT NULL, componente TEXT NOT NULL, cadastro TEXT NOT NULL,\r\n  gap_min REAL, overlap_min REAL, start_delta_min REAL,\r\n  duration_comparable INTEGER NOT NULL, duration_abs_min REAL, duration_ratio REAL, duration_relevant INTEGER,\r\n  sap_sem_fim INTEGER NOT NULL, janela_sap_longa INTEGER NOT NULL, score_v3 INTEGER,\r\n  signals_json TEXT NOT NULL CHECK(json_valid(signals_json)),\r\n  PRIMARY KEY(dataset_id, rules_version, mes_id, sap_id),\r\n  FOREIGN KEY(dataset_id, rules_version) REFERENCES ev_rule_version(dataset_id, rules_version),\r\n  FOREIGN KEY(dataset_id, mes_id, mes_system) REFERENCES source_record(dataset_id, record_id, source_system),\r\n  FOREIGN KEY(dataset_id, sap_id, sap_system) REFERENCES source_record(dataset_id, record_id, source_system),\r\n  FOREIGN KEY(dataset_id, mes_id, sap_id) REFERENCES association_candidate(dataset_id, mes_id, sap_id)\r\n);\r\nCREATE INDEX ix_ev_pair_sap ON ev_pair_signal(dataset_id, rules_version, sap_id, classe);\r\nCREATE TABLE ev_block (\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL, block_id TEXT NOT NULL, machine TEXT,\r\n  n_mes INTEGER NOT NULL, n_sap INTEGER NOT NULL, n_pairs INTEGER NOT NULL,\r\n  label TEXT NOT NULL,\r\n  PRIMARY KEY(dataset_id, rules_version, block_id),\r\n  FOREIGN KEY(dataset_id, rules_version) REFERENCES ev_rule_version(dataset_id, rules_version)\r\n);\r\nCREATE TABLE ev_block_member (\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL, block_id TEXT NOT NULL,\r\n  record_id TEXT NOT NULL, source_system TEXT NOT NULL CHECK(source_system IN ('MES','SAP')),\r\n  PRIMARY KEY(dataset_id, rules_version, record_id),\r\n  FOREIGN KEY(dataset_id, rules_version, block_id) REFERENCES ev_block(dataset_id, rules_version, block_id),\r\n  FOREIGN KEY(dataset_id, record_id, source_system) REFERENCES source_record(dataset_id, record_id, source_system)\r\n);\r\nCREATE TABLE ev_record_decision (\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL,\r\n  record_id TEXT NOT NULL, source_system TEXT NOT NULL CHECK(source_system IN ('MES','SAP')),\r\n  decisao TEXT NOT NULL CHECK(decisao IN ('sem_candidato','correspondencia_provavel','disputa','evidencia_insuficiente')),\r\n  classe_maxima TEXT CHECK(classe_maxima IS NULL OR classe_maxima IN ('forte','moderada','fraca')),\r\n  contraparte_id TEXT, n_candidatos INTEGER NOT NULL, n_forte INTEGER NOT NULL, n_moderada INTEGER NOT NULL, n_fraca INTEGER NOT NULL,\r\n  cadastro TEXT NOT NULL, intervalo_incompleto INTEGER NOT NULL, alternativa_incompleta INTEGER NOT NULL,\r\n  revisao TEXT NOT NULL CHECK(revisao='pendente'),\r\n  motivos_json TEXT NOT NULL CHECK(json_valid(motivos_json)),\r\n  motivos_confianca_json TEXT NOT NULL CHECK(json_valid(motivos_confianca_json)),\r\n  saidas_json TEXT NOT NULL CHECK(json_valid(saidas_json)),\r\n  alternativas_json TEXT NOT NULL CHECK(json_valid(alternativas_json)),\r\n  block_id TEXT,\r\n  PRIMARY KEY(dataset_id, rules_version, record_id),\r\n  FOREIGN KEY(dataset_id, rules_version) REFERENCES ev_rule_version(dataset_id, rules_version),\r\n  FOREIGN KEY(dataset_id, record_id, source_system) REFERENCES source_record(dataset_id, record_id, source_system),\r\n  FOREIGN KEY(dataset_id, rules_version, block_id) REFERENCES ev_block(dataset_id, rules_version, block_id),\r\n  CHECK((decisao='sem_candidato' AND n_candidatos=0) OR (decisao<>'sem_candidato' AND n_candidatos>0))\r\n);\r\nCREATE INDEX ix_ev_decision ON ev_record_decision(dataset_id, rules_version, source_system, decisao);\r\nCREATE TABLE ev_hypothesis (\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL, hypothesis_id TEXT NOT NULL, block_id TEXT NOT NULL,\r\n  tipo TEXT NOT NULL CHECK(tipo IN ('1:1','N:1','1:N','N:M')),\r\n  estado TEXT NOT NULL CHECK(estado IN ('proposta','nm_nao_enumerado')),\r\n  incompleto INTEGER NOT NULL, duracao_json TEXT NOT NULL CHECK(json_valid(duracao_json)),\r\n  uniao_json TEXT NOT NULL CHECK(json_valid(uniao_json)), alternativas_json TEXT NOT NULL CHECK(json_valid(alternativas_json)),\r\n  PRIMARY KEY(dataset_id, rules_version, hypothesis_id),\r\n  FOREIGN KEY(dataset_id, rules_version, block_id) REFERENCES ev_block(dataset_id, rules_version, block_id)\r\n);\r\nCREATE TABLE ev_hypothesis_member (\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL, hypothesis_id TEXT NOT NULL,\r\n  record_id TEXT NOT NULL, source_system TEXT NOT NULL CHECK(source_system IN ('MES','SAP')),\r\n  PRIMARY KEY(dataset_id, rules_version, hypothesis_id, record_id),\r\n  FOREIGN KEY(dataset_id, rules_version, hypothesis_id) REFERENCES ev_hypothesis(dataset_id, rules_version, hypothesis_id),\r\n  FOREIGN KEY(dataset_id, record_id, source_system) REFERENCES source_record(dataset_id, record_id, source_system)\r\n);\r\nCREATE TABLE ev_flag (\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL, flag_id TEXT NOT NULL,\r\n  tipo TEXT NOT NULL CHECK(tipo IN ('duplicata_exata','possivel_duplicidade','possivel_fragmentacao')),\r\n  source_system TEXT NOT NULL CHECK(source_system IN ('MES','SAP')),\r\n  regra TEXT NOT NULL, campos_json TEXT NOT NULL CHECK(json_valid(campos_json)), estado TEXT NOT NULL,\r\n  PRIMARY KEY(dataset_id, rules_version, flag_id),\r\n  FOREIGN KEY(dataset_id, rules_version) REFERENCES ev_rule_version(dataset_id, rules_version)\r\n);\r\nCREATE TABLE ev_flag_member (\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL, flag_id TEXT NOT NULL,\r\n  record_id TEXT NOT NULL, source_system TEXT NOT NULL CHECK(source_system IN ('MES','SAP')),\r\n  PRIMARY KEY(dataset_id, rules_version, flag_id, record_id),\r\n  FOREIGN KEY(dataset_id, rules_version, flag_id) REFERENCES ev_flag(dataset_id, rules_version, flag_id),\r\n  FOREIGN KEY(dataset_id, record_id, source_system) REFERENCES source_record(dataset_id, record_id, source_system)\r\n);\r\nCREATE TABLE ev_review (\r\n  review_id INTEGER PRIMARY KEY AUTOINCREMENT,\r\n  dataset_id TEXT NOT NULL, rules_version TEXT NOT NULL,\r\n  record_id TEXT NOT NULL, source_system TEXT NOT NULL CHECK(source_system IN ('MES','SAP')),\r\n  responsavel TEXT NOT NULL, data TEXT NOT NULL, motivo TEXT NOT NULL,\r\n  estado TEXT NOT NULL CHECK(estado IN ('pendente','confirmada','rejeitada','incerta')),\r\n  FOREIGN KEY(dataset_id, rules_version) REFERENCES ev_rule_version(dataset_id, rules_version),\r\n  FOREIGN KEY(dataset_id, record_id, source_system) REFERENCES source_record(dataset_id, record_id, source_system)\r\n);\r\nCREATE VIEW ev_v3_comparison AS\r\nSELECT v.dataset_id, v.rules_version, d.mes_id, d.status AS v3_status, d.score AS v3_best_score,\r\n       d.candidate_count AS v3_candidate_count, e.n_candidatos AS ev_candidate_count,\r\n       e.decisao AS ev_decisao, e.classe_maxima AS ev_classe_maxima, e.contraparte_id AS ev_contraparte_id,\r\n       d.suggested_sap_id AS v3_suggested_sap_id\r\nFROM ev_rule_version AS v\r\nJOIN association_decision AS d ON d.dataset_id=v.dataset_id\r\nJOIN ev_record_decision AS e ON e.dataset_id=v.dataset_id AND e.rules_version=v.rules_version\r\n  AND e.record_id=d.mes_id AND e.source_system='MES';\r\n'''\r\n\r\n\r\ndef history_like(mes_rows, sap_rows):\r\n    \"\"\"Monta um histórico mínimo (uso em testes sintéticos).\"\"\"\r\n    return {'schemaVersion': 1, 'datasetVersion': 'sintetico-1', 'mes': mes_rows, 'sap': sap_rows,\r\n            'metadata': {'sourceHashes': [], 'generatedAt': '2026-01-01T00:00:00+00:00', 'pipelineSha256': None},\r\n            'association': associate(mes_rows, sap_rows)}\r\n\r\n\r\ndef build_snapshot(history, input_sha256, window=60):\r\n    result = ce.analyze_history(history, window)\r\n    body = {\r\n        'schemaVersion': 1, 'kind': 'sime-sp2-evidencias', 'rulesVersion': ce.RULES_VERSION, 'rules': ce.RULES, 'window': window,\r\n        'origin': {'inputSha256': input_sha256, 'datasetVersion': history.get('datasetVersion'),\r\n                   'v3AlgorithmVersion': history['association']['algorithmVersion'],\r\n                   'sourceHashes': history['metadata'].get('sourceHashes'),\r\n                   'mes': len(history['mes']), 'sap': len(history['sap'])},\r\n        'result': result,\r\n        'reviews': [],\r\n        'nota': 'Método candidato em paralelo à v3. Classes são ordinais, não probabilidades; nada aqui confirma evento físico, '\r\n                'valida cadastro ou representa revisão humana.',\r\n    }\r\n    body['contentSha256'] = ce.content_hash(body)\r\n    return body\r\n\r\n\r\ndef compact_for_history(snapshot):\r\n    \"\"\"Subconjunto aditivo para `associationCandidate`: dados por registro, textos das regras vão uma só vez em `rules`.\"\"\"\r\n    result = snapshot['result']\r\n    decisions = []\r\n    for d in result['decisions']:\r\n        row = {k: d[k] for k in ('id', 'sistema', 'decisao', 'classeMaxima', 'contraparteId', 'nCandidatos', 'nForte', 'nModerada',\r\n                                  'nFraca', 'motivos', 'motivosConfianca', 'cadastro', 'intervaloIncompleto',\r\n                                  'alternativaIncompleta', 'revisao', 'blocoId', 'saidas', 'hipoteseIds', 'flagIds')}\r\n        row['alternativas'] = [{'id': a['id'], 'classe': a['classe']} for a in d['alternativas']]\r\n        decisions.append(row)\r\n    pairs = [{k: p[k] for k in ('mesId', 'sapId', 'classe', 'temporal', 'gapMin', 'sobreposicaoMin', 'deltaInicioMin', 'deltaFimMin',\r\n                                 'subdivisao', 'componente', 'tokensComponente', 'cadastro', 'duracao', 'sapSemFim',\r\n                                 'janelaSapLonga', 'scoreV3')} for p in result['pairs']]\r\n    hypotheses = []\r\n    for h in result['hypotheses']:\r\n        row = {k: h[k] for k in ('id', 'tipo', 'estado', 'blocoId', 'mesIds', 'sapIds', 'alternativas')}\r\n        if h['tipo'] != '1:1':\r\n            row.update({k: h[k] for k in ('pares', 'uniao', 'janelaSapLonga', 'incompleto', 'duracao')})\r\n        hypotheses.append(row)\r\n    flags = [{'id': f['id'], 'tipo': f['tipo'], 'sistema': f['sistema'], 'registros': f['registros'], 'campos': f['campos']}\r\n             for f in result['flags']]\r\n    blocks = [{k: b[k] for k in ('id', 'maquina', 'mesIds', 'sapIds', 'nPares', 'hipoteseIds')} for b in result['blocks']]\r\n    return {'schemaVersion': 1, 'rulesVersion': snapshot['rulesVersion'], 'rules': snapshot['rules'], 'window': snapshot['window'],\r\n            'origin': {'inputSha256': snapshot['origin']['inputSha256'], 'snapshotContentSha256': snapshot['contentSha256'],\r\n                       'v3AlgorithmVersion': snapshot['origin']['v3AlgorithmVersion']},\r\n            'nota': snapshot['nota'], 'summary': result['summary'], 'decisions': decisions, 'pairs': pairs, 'blocks': blocks,\r\n            'hypotheses': hypotheses, 'flags': flags, 'reviews': []}\r\n\r\n\r\ndef _write_new(path, text):\r\n    path = Path(path)\r\n    path.parent.mkdir(parents=True, exist_ok=True)\r\n    with open(path, 'x', encoding='utf-8', newline='\\n') as handle:\r\n        handle.write(text)\r\n\r\n\r\ndef write_database(history, snapshot, database_path):\r\n    \"\"\"SQLite com o esquema v3 (`SCHEMA`) mais as tabelas aditivas `ev_*`, na mesma transação.\"\"\"\r\n    output = Path(database_path)\r\n    output.parent.mkdir(parents=True, exist_ok=True)\r\n    if output.exists():\r\n        raise FileExistsError(f'Banco já existe; escolha novo caminho para preservar a versão: {output}')\r\n    association = history['association']\r\n    algorithm = association['algorithmVersion']\r\n    manifest = history['metadata'].get('sourceHashes')\r\n    input_sha = snapshot['origin']['inputSha256']\r\n    dataset_id = 'sp2-' + hashlib.sha256(canonical({'datasetVersion': history['datasetVersion'], 'algorithm': algorithm,\r\n                                                     'sources': manifest, 'pipelineSha256': history['metadata'].get('pipelineSha256'),\r\n                                                     'contentSha256': input_sha}).encode('utf-8')).hexdigest()[:20]\r\n    rules_version, result = snapshot['rulesVersion'], snapshot['result']\r\n    key = (dataset_id, rules_version)\r\n    temp = output.with_name(f'.{output.name}.{uuid4().hex}.tmp')\r\n    db = sqlite3.connect(temp)\r\n    try:\r\n        with db:\r\n            db.executescript(SCHEMA)\r\n            db.executescript(EV_SCHEMA)\r\n            db.execute('INSERT INTO dataset_version VALUES (?,?,?,?,?,?,?,?,?)', (\r\n                dataset_id, history['datasetVersion'], algorithm, input_sha, history['metadata']['generatedAt'],\r\n                canonical(manifest), canonical(history['metadata']), len(history['mes']), len(history['sap'])))\r\n            records = []\r\n            for system, rows in (('MES', history['mes']), ('SAP', history['sap'])):\r\n                for row in rows:\r\n                    source = row['source']\r\n                    normalized = {k: row.get(k) for k in ('machine', 'family', 'subdivision', 'component', 'type', 'activity', 'text',\r\n                                                          'start', 'end', 'durationMinutes', 'mappingStatus') if k in row}\r\n                    records.append((dataset_id, row['id'], system, row['machine'], source['file'], source['sheet'], source['row'],\r\n                                    source['hash'], row.get('start'), row.get('end'), row.get('durationMinutes'),\r\n                                    canonical(row.get('raw', {})), canonical(normalized), canonical(row.get('qualityFlags', []))))\r\n            db.executemany('INSERT INTO source_record VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)', records)\r\n            db.executemany('INSERT INTO association_candidate VALUES (?,?,?,?,?,?,?)',\r\n                           [(dataset_id, r['mesId'], 'MES', r['sapId'], 'SAP', r['score'], canonical(r)) for r in association['candidates']])\r\n            db.executemany('INSERT INTO association_decision VALUES (?,?,?,?,?,?,?,?,?,?)',\r\n                           [(dataset_id, r['mesId'], 'MES', r['status'], r['selectedSapId'], r.get('suggestedSapId'), r['score'],\r\n                             r['reason'], r['algorithmVersion'], r['candidateCount']) for r in association['decisions']])\r\n            db.execute('INSERT INTO ev_rule_version VALUES (?,?,?,?,?,?)',\r\n                       (*key, snapshot['window'], input_sha, snapshot['contentSha256'], canonical(snapshot['rules'])))\r\n            db.executemany('INSERT INTO ev_block VALUES (?,?,?,?,?,?,?,?)',\r\n                           [(*key, b['id'], b['maquina'], len(b['mesIds']), len(b['sapIds']), b['nPares'], b['rotulo']) for b in result['blocks']])\r\n            db.executemany('INSERT INTO ev_block_member VALUES (?,?,?,?,?)',\r\n                           [(*key, b['id'], i, 'MES') for b in result['blocks'] for i in b['mesIds']] +\r\n                           [(*key, b['id'], i, 'SAP') for b in result['blocks'] for i in b['sapIds']])\r\n            db.executemany('INSERT INTO ev_pair_signal VALUES (' + ','.join('?' * 22) + ')', [\r\n                (*key, p['mesId'], 'MES', p['sapId'], 'SAP', p['classe'], p['temporal'], p['subdivisao'], p['componente'], p['cadastro'],\r\n                 p['gapMin'], p['sobreposicaoMin'], p['deltaInicioMin'], int(p['duracao']['comparavel']), p['duracao']['absolutaMin'],\r\n                 p['duracao']['razao'], None if p['duracao']['relevante'] is None else int(p['duracao']['relevante']),\r\n                 int(p['sapSemFim']), int(p['janelaSapLonga']), p['scoreV3'], canonical(p)) for p in result['pairs']])\r\n            db.executemany('INSERT INTO ev_record_decision VALUES (' + ','.join('?' * 20) + ')', [\r\n                (*key, d['id'], d['sistema'], d['decisao'], d['classeMaxima'], d['contraparteId'], d['nCandidatos'], d['nForte'],\r\n                 d['nModerada'], d['nFraca'], d['cadastro'], int(d['intervaloIncompleto']), int(d['alternativaIncompleta']), d['revisao'],\r\n                 canonical(d['motivos']), canonical(d['motivosConfianca']), canonical(d['saidas']), canonical(d['alternativas']),\r\n                 d['blocoId']) for d in result['decisions']])\r\n            db.executemany('INSERT INTO ev_hypothesis VALUES (?,?,?,?,?,?,?,?,?,?)', [\r\n                (*key, h['id'], h['blocoId'], h['tipo'], h['estado'], int(h['incompleto']), canonical(h['duracao']),\r\n                 canonical(h['uniao']), canonical(h['alternativas'])) for h in result['hypotheses']])\r\n            db.executemany('INSERT INTO ev_hypothesis_member VALUES (?,?,?,?,?)',\r\n                           [(*key, h['id'], i, 'MES') for h in result['hypotheses'] for i in h['mesIds']] +\r\n                           [(*key, h['id'], i, 'SAP') for h in result['hypotheses'] for i in h['sapIds']])\r\n            db.executemany('INSERT INTO ev_flag VALUES (?,?,?,?,?,?,?,?)', [\r\n                (*key, f['id'], f['tipo'], f['sistema'], f['regra'], canonical(f['campos']), f['estado']) for f in result['flags']])\r\n            db.executemany('INSERT INTO ev_flag_member VALUES (?,?,?,?,?)',\r\n                           [(*key, f['id'], i, f['sistema']) for f in result['flags'] for i in f['registros']])\r\n            integrity = db.execute('PRAGMA integrity_check').fetchone()[0]\r\n            fk_errors = db.execute('PRAGMA foreign_key_check').fetchall()\r\n            decisions = db.execute('SELECT COUNT(*), COUNT(DISTINCT record_id) FROM ev_record_decision').fetchone()\r\n            if integrity != 'ok' or fk_errors or decisions[0] != decisions[1] or decisions[0] != len(records):\r\n                raise RuntimeError(f'Banco inválido: integrity={integrity}, fk={len(fk_errors)}, decisões={decisions}')\r\n            summary = {'datasetId': dataset_id, 'rulesVersion': rules_version, 'integrity': integrity, 'foreignKeyErrors': len(fk_errors),\r\n                       'sourceRecords': len(records), 'v3Candidates': len(association['candidates']),\r\n                       'evidencePairs': len(result['pairs']), 'evidenceDecisions': decisions[0], 'blocks': len(result['blocks']),\r\n                       'hypotheses': len(result['hypotheses']), 'flags': len(result['flags']), 'reviews': 0}\r\n        db.close()\r\n        temp.replace(output)\r\n        summary['databaseBytes'] = output.stat().st_size\r\n        return summary\r\n    except Exception:\r\n        db.close()\r\n        temp.unlink(missing_ok=True)\r\n        raise\r\n\r\n\r\ndef main():\r\n    parser = argparse.ArgumentParser(description=__doc__)\r\n    parser.add_argument('--input', required=True, help='histórico JSON existente (v3 embutida)')\r\n    parser.add_argument('--snapshot-output', required=True)\r\n    parser.add_argument('--db-output', required=True)\r\n    parser.add_argument('--history-output', required=True, help='histórico original + associationCandidate (novo arquivo)')\r\n    parser.add_argument('--window', type=int, default=60)\r\n    args = parser.parse_args()\r\n    raw = Path(args.input).read_bytes()\r\n    history = json.loads(raw)\r\n    input_sha = hashlib.sha256(raw).hexdigest()\r\n    snapshot = build_snapshot(history, input_sha, args.window)\r\n    outputs = [Path(p) for p in (args.snapshot_output, args.db_output, args.history_output)]\r\n    for path in outputs:\r\n        if path.exists():\r\n            raise FileExistsError(f'Recusando sobrescrever: {path}')\r\n    db_summary = write_database(history, snapshot, args.db_output)\r\n    stamped = dict(snapshot, generatedAt=datetime.now(timezone.utc).isoformat())\r\n    _write_new(args.snapshot_output, json.dumps(stamped, ensure_ascii=False, indent=1) + '\\n')\r\n    additive = dict(history)\r\n    additive['associationCandidate'] = compact_for_history(snapshot)\r\n    _write_new(args.history_output, json.dumps(additive, ensure_ascii=False, separators=(',', ':')) + '\\n')\r\n    print(json.dumps({'inputSha256': input_sha, 'contentSha256': snapshot['contentSha256'], 'database': db_summary,\r\n                      'summary': snapshot['result']['summary']}, ensure_ascii=False))\r\n\r\n\r\nif __name__ == '__main__':\r\n    main()\r\n",
 "05_subprodutos/02_conciliacao/criar_base_consolidada.py": "\"\"\"Cria uma base SQLite versionada, normalizada e reproduzível para o SP2.\"\"\"\nimport argparse\nimport hashlib\nimport json\nimport sqlite3\nfrom pathlib import Path\nfrom uuid import uuid4\n\n\nSCHEMA = '''\nPRAGMA foreign_keys=ON;\nCREATE TABLE dataset_version (\n  dataset_id TEXT PRIMARY KEY,\n  dataset_version TEXT NOT NULL,\n  algorithm_version TEXT NOT NULL,\n  content_sha256 TEXT NOT NULL CHECK(length(content_sha256)=64),\n  generated_at TEXT NOT NULL,\n  source_hashes_json TEXT NOT NULL CHECK(json_valid(source_hashes_json)),\n  metadata_json TEXT NOT NULL CHECK(json_valid(metadata_json)),\n  mes_count INTEGER NOT NULL CHECK(mes_count >= 0),\n  sap_count INTEGER NOT NULL CHECK(sap_count >= 0)\n);\nCREATE TABLE source_record (\n  dataset_id TEXT NOT NULL,\n  record_id TEXT NOT NULL,\n  source_system TEXT NOT NULL CHECK(source_system IN ('MES','SAP')),\n  asset_id TEXT NOT NULL,\n  source_file TEXT NOT NULL,\n  source_sheet TEXT NOT NULL,\n  source_row INTEGER NOT NULL CHECK(source_row > 1),\n  source_sha256 TEXT NOT NULL CHECK(length(source_sha256)=64),\n  starts_at TEXT,\n  ends_at TEXT,\n  duration_minutes REAL CHECK(duration_minutes IS NULL OR duration_minutes >= 0),\n  original_json TEXT NOT NULL CHECK(json_valid(original_json)),\n  normalized_json TEXT NOT NULL CHECK(json_valid(normalized_json)),\n  quality_flags_json TEXT NOT NULL CHECK(json_valid(quality_flags_json)),\n  PRIMARY KEY(dataset_id, record_id),\n  UNIQUE(dataset_id, record_id, source_system),\n  UNIQUE(dataset_id, source_system, source_file, source_sheet, source_row),\n  FOREIGN KEY(dataset_id) REFERENCES dataset_version(dataset_id)\n);\nCREATE INDEX ix_source_asset_time ON source_record(dataset_id, asset_id, starts_at);\nCREATE INDEX ix_source_system_asset ON source_record(dataset_id, source_system, asset_id);\nCREATE TABLE association_candidate (\n  dataset_id TEXT NOT NULL,\n  mes_id TEXT NOT NULL,\n  mes_system TEXT NOT NULL DEFAULT 'MES' CHECK(mes_system='MES'),\n  sap_id TEXT NOT NULL,\n  sap_system TEXT NOT NULL DEFAULT 'SAP' CHECK(sap_system='SAP'),\n  score INTEGER NOT NULL CHECK(score BETWEEN 0 AND 100),\n  evidence_json TEXT NOT NULL CHECK(json_valid(evidence_json)),\n  PRIMARY KEY(dataset_id, mes_id, sap_id),\n  FOREIGN KEY(dataset_id, mes_id, mes_system) REFERENCES source_record(dataset_id, record_id, source_system),\n  FOREIGN KEY(dataset_id, sap_id, sap_system) REFERENCES source_record(dataset_id, record_id, source_system)\n);\nCREATE INDEX ix_candidate_sap ON association_candidate(dataset_id, sap_id, score DESC);\nCREATE TABLE association_decision (\n  dataset_id TEXT NOT NULL,\n  mes_id TEXT NOT NULL,\n  mes_system TEXT NOT NULL DEFAULT 'MES' CHECK(mes_system='MES'),\n  status TEXT NOT NULL CHECK(status IN ('auto_associated_unvalidated','suggested_unvalidated_mapping','ambiguous','review_required','review_incomplete_interval','no_candidate')),\n  selected_sap_id TEXT,\n  suggested_sap_id TEXT,\n  score INTEGER CHECK(score IS NULL OR score BETWEEN 0 AND 100),\n  reason TEXT NOT NULL,\n  algorithm_version TEXT NOT NULL,\n  candidate_count INTEGER NOT NULL CHECK(candidate_count >= 0),\n  PRIMARY KEY(dataset_id, mes_id),\n  FOREIGN KEY(dataset_id, mes_id, mes_system) REFERENCES source_record(dataset_id, record_id, source_system),\n  FOREIGN KEY(dataset_id, mes_id, selected_sap_id) REFERENCES association_candidate(dataset_id, mes_id, sap_id),\n  FOREIGN KEY(dataset_id, mes_id, suggested_sap_id) REFERENCES association_candidate(dataset_id, mes_id, sap_id),\n  CHECK((status='auto_associated_unvalidated' AND selected_sap_id IS NOT NULL) OR\n        (status<>'auto_associated_unvalidated' AND selected_sap_id IS NULL)),\n  CHECK((status='suggested_unvalidated_mapping' AND suggested_sap_id IS NOT NULL) OR\n        (status<>'suggested_unvalidated_mapping' AND suggested_sap_id IS NULL))\n);\nCREATE INDEX ix_decision_status ON association_decision(dataset_id, status);\nCREATE VIEW association_joined AS\nSELECT m.dataset_id, m.record_id AS mes_id, m.asset_id,\n       m.starts_at AS mes_starts_at, m.ends_at AS mes_ends_at,\n       d.status AS association_status, d.score AS association_score,\n       d.selected_sap_id, d.suggested_sap_id,\n       s.record_id AS suggested_or_selected_sap_record_id,\n       s.starts_at AS candidate_sap_starts_at, s.ends_at AS candidate_sap_ends_at,\n       s.normalized_json AS candidate_sap_normalized_json\nFROM source_record AS m\nJOIN association_decision AS d ON d.dataset_id=m.dataset_id AND d.mes_id=m.record_id\nLEFT JOIN source_record AS s ON s.dataset_id=m.dataset_id\n  AND s.record_id=COALESCE(d.selected_sap_id, d.suggested_sap_id)\n  AND s.source_system='SAP'\nWHERE m.source_system='MES';\n'''\n\n\ndef canonical(value):\n    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(',', ':'))\n\n\ndef load_data(path):\n    data = json.loads(Path(path).read_text(encoding='utf-8'))\n    if data.get('schemaVersion') != 1 or not isinstance(data.get('association'), dict):\n        raise ValueError('Histórico precisa ter schemaVersion=1 e association v2.')\n    return data\n\n\ndef create_database(source_path, database_path):\n    data = load_data(source_path)\n    source_manifest = data['metadata']['sourceHashes']\n    algorithm = data['association']['algorithmVersion']\n    candidate_counts = {}\n    for candidate in data['association']['candidates']:\n        candidate_counts[candidate['mesId']] = candidate_counts.get(candidate['mesId'], 0) + 1\n    if any(row['candidateCount'] != candidate_counts.get(row['mesId'], 0) for row in data['association']['decisions']):\n        raise ValueError('candidateCount diverge dos pares candidatos por registro MES.')\n    actual_statuses = {}\n    for row in data['association']['decisions']:\n        actual_statuses[row['status']] = actual_statuses.get(row['status'], 0) + 1\n    if actual_statuses != {key: value for key, value in data['association']['summary'].items() if value > 0}:\n        raise ValueError('Resumo de decisões diverge da contagem de status.')\n    content_sha256 = hashlib.sha256(Path(source_path).read_bytes()).hexdigest()\n    dataset_seed = canonical({'datasetVersion': data['datasetVersion'], 'algorithm': algorithm,\n                              'sources': source_manifest, 'pipelineSha256': data['metadata'].get('pipelineSha256'),\n                              'contentSha256': content_sha256})\n    dataset_id = 'sp2-' + hashlib.sha256(dataset_seed.encode('utf-8')).hexdigest()[:20]\n    output = Path(database_path)\n    output.parent.mkdir(parents=True, exist_ok=True)\n    if output.exists():\n        raise FileExistsError(f'Banco já existe; escolha novo caminho para preservar a versão: {output}')\n    temp = output.with_name(f'.{output.name}.{uuid4().hex}.tmp')\n    db = sqlite3.connect(temp)\n    try:\n        with db:\n            db.executescript(SCHEMA)\n            db.execute('INSERT INTO dataset_version VALUES (?,?,?,?,?,?,?,?,?)', (\n                dataset_id, data['datasetVersion'], algorithm, content_sha256, data['metadata']['generatedAt'],\n                canonical(source_manifest), canonical(data['metadata']), len(data['mes']), len(data['sap'])))\n            records = []\n            for system, rows in (('MES', data['mes']), ('SAP', data['sap'])):\n                for row in rows:\n                    source = row['source']\n                    normalized = {k: row.get(k) for k in ('machine','family','subdivision','component','type','activity','text','start','end','durationMinutes','mappingStatus') if k in row}\n                    records.append((dataset_id, row['id'], system, row['machine'], source['file'], source['sheet'],\n                        source['row'], source['hash'], row.get('start'), row.get('end'), row.get('durationMinutes'),\n                        canonical(row.get('raw', {})), canonical(normalized), canonical(row.get('qualityFlags', []))))\n            db.executemany('INSERT INTO source_record VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)', records)\n            candidates = [(dataset_id, row['mesId'], 'MES', row['sapId'], 'SAP', row['score'], canonical(row))\n                          for row in data['association']['candidates']]\n            db.executemany('INSERT INTO association_candidate VALUES (?,?,?,?,?,?,?)', candidates)\n            decisions = [(dataset_id, row['mesId'], 'MES', row['status'], row['selectedSapId'], row.get('suggestedSapId'), row['score'],\n                          row['reason'], row['algorithmVersion'], row['candidateCount'])\n                         for row in data['association']['decisions']]\n            db.executemany('INSERT INTO association_decision VALUES (?,?,?,?,?,?,?,?,?,?)', decisions)\n            integrity = db.execute('PRAGMA integrity_check').fetchone()[0]\n            fk_errors = db.execute('PRAGMA foreign_key_check').fetchall()\n            if integrity != 'ok' or fk_errors:\n                raise RuntimeError(f'Banco inválido: integrity={integrity}, foreignKeys={len(fk_errors)}')\n            summary = {\n                'datasetId': dataset_id, 'algorithmVersion': algorithm, 'integrity': integrity,\n                'foreignKeyErrors': len(fk_errors), 'sourceRecords': len(records),\n                'candidates': len(candidates), 'decisions': len(decisions),\n            'autoLinks': sum(row[3] == 'auto_associated_unvalidated' for row in decisions),\n            'autoSuggestions': sum(row[5] is not None for row in decisions),\n                'databaseBytes': temp.stat().st_size,\n            }\n        db.close()\n        temp.replace(output)\n        summary['databaseBytes'] = output.stat().st_size\n        return summary\n    except Exception:\n        db.close()\n        # Keep a failed DB for diagnosis; never replace or delete it automatically.\n        raise\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument('--input', required=True, help='JSON do histórico gerado com o algoritmo SP2 v2.')\n    parser.add_argument('--output', required=True, help='Caminho novo para o banco SQLite.')\n    args = parser.parse_args()\n    result = create_database(args.input, args.output)\n    print(json.dumps(result, ensure_ascii=False))\n\n\nif __name__ == '__main__':\n    main()\n",
 "05_subprodutos/02_conciliacao/test_associacao_automatica.py": "import sys\nimport unittest\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).parent))\nfrom associacao_automatica import associate, POLICY\n\n\ndef mes(record_id='MES-R0001', machine='A', subdivision='DISCO', component='QMD', start='2026-01-01T10:00:00', end='2026-01-01T10:30:00'):\n    return {'id': record_id, 'machine': machine, 'subdivision': subdivision, 'component': component,\n            'start': start, 'end': end}\n\n\ndef sap(record_id='SAP-R0001', machine='A', subdivision='DISCO', text='FALHA QMD', start='2026-01-01T10:00:00', end='2026-01-01T10:30:00'):\n    return {'id': record_id, 'machine': machine, 'subdivision': subdivision, 'text': text,\n            'start': start, 'end': end, 'mappingStatus': 'validado tecnicamente', 'assetMappingValidated': True}\n\n\nclass AutomaticAssociationTests(unittest.TestCase):\n    def test_strong_unique_pair_is_joined_automatically_with_provenance(self):\n        result = associate([mes()], [sap()])\n        decision = result['decisions'][0]\n        self.assertEqual(decision['status'], 'auto_associated_unvalidated')\n        self.assertEqual(decision['selectedSapId'], 'SAP-R0001')\n        self.assertEqual(result['automaticLinks'][0]['algorithmVersion'], 'sp2-association-v3')\n        self.assertFalse(POLICY['scoreIsProbability'])\n\n    def test_equal_competing_candidates_abstain_as_ambiguous(self):\n        result = associate([mes()], [sap('SAP-R0001'), sap('SAP-R0002')])\n        self.assertEqual(result['decisions'][0]['status'], 'ambiguous')\n        self.assertIsNone(result['decisions'][0]['selectedSapId'])\n\n    def test_incomplete_sap_interval_never_auto_associates(self):\n        result = associate([mes()], [sap(end=None)])\n        self.assertEqual(result['decisions'][0]['status'], 'review_incomplete_interval')\n        self.assertEqual(result['automaticLinks'], [])\n\n    def test_different_asset_has_no_candidate(self):\n        result = associate([mes(machine='A')], [sap(machine='B')])\n        self.assertEqual(result['decisions'][0]['status'], 'no_candidate')\n\n    def test_far_apart_intervals_are_not_joined(self):\n        result = associate([mes()], [sap(start='2026-01-01T13:00:00', end='2026-01-01T13:30:00')])\n        self.assertEqual(result['decisions'][0]['status'], 'no_candidate')\n\n    def test_inferred_asset_mapping_only_creates_automatic_suggestion(self):\n        result = associate([mes()], [sap() | {'mappingStatus': 'validado tecnicamente', 'assetMappingValidated': False}])\n        decision = result['decisions'][0]\n        self.assertEqual(decision['status'], 'suggested_unvalidated_mapping')\n        self.assertIsNone(decision['selectedSapId'])\n        self.assertEqual(decision['suggestedSapId'], 'SAP-R0001')\n        self.assertEqual(result['automaticLinks'], [])\n        self.assertEqual(result['automaticSuggestions'][0]['sapId'], 'SAP-R0001')\n\n    def test_nearby_times_do_not_resolve_two_mes_records_competing_for_same_sap(self):\n        first, second = mes('MES-R0001'), mes('MES-R0002', start='2026-01-01T10:01:00', end='2026-01-01T10:31:00')\n        result = associate([first, second], [sap()])\n        self.assertEqual(result['automaticLinks'], [])\n        self.assertTrue(all(row['selectedSapId'] is None for row in result['decisions']))\n\n    def test_invalid_or_reversed_intervals_fail_closed(self):\n        with self.assertRaisesRegex(ValueError, 'fim anterior'):\n            associate([mes(start='2026-01-01T10:30:00', end='2026-01-01T10:00:00')], [sap()])\n        with self.assertRaisesRegex(ValueError, 'Timestamp ISO inválido'):\n            associate([mes(start='ontem')], [sap()])\n\n    def test_mapping_validation_requires_an_explicit_boolean(self):\n        result = associate([mes()], [sap() | {'assetMappingValidated': 'true'}])\n        self.assertEqual(result['decisions'][0]['status'], 'suggested_unvalidated_mapping')\n\n    def test_incomplete_candidate_beyond_top_three_is_visible_for_review(self):\n        saps = [sap(f'SAP-{i:04}', start=f'2026-01-01T10:{i:02}:00', end=f'2026-01-01T10:{i+1:02}:00')\n                for i in range(1, 5)]\n        saps.append(sap('SAP-INCOMPLETE', start='2026-01-01T10:10:00', end=None))\n        result = associate([mes()], saps)\n        self.assertEqual(result['decisions'][0]['status'], 'review_incomplete_interval')\n        self.assertTrue(any(not c['completeSapInterval'] for c in result['candidates']))\n\n\nif __name__ == '__main__':\n    unittest.main()\n",
 "05_subprodutos/02_conciliacao/test_conciliacao_evidencias.py": "\"\"\"Testes SINTÉTICOS do método `sp2-evidence-v1`. Rótulos e cenários são artificiais: não validam acurácia industrial.\"\"\"\r\nimport copy\r\nimport json\r\nimport sqlite3\r\nimport tempfile\r\nimport unittest\r\nfrom datetime import datetime, timedelta\r\nfrom pathlib import Path\r\n\r\nimport associacao_automatica as v3\r\nimport conciliacao_evidencias as ce\r\n\r\nBASE = datetime(2026, 3, 1, 8, 0, 0)\r\n\r\n\r\ndef stamp(minutes):\r\n    return (BASE + timedelta(minutes=minutes)).isoformat()\r\n\r\n\r\ndef mes(rid, start, end, machine='A', sub='FORNO 1', comp='BCL', raw=None):\r\n    return {'id': rid, 'machine': machine, 'subdivision': sub, 'component': comp, 'start': stamp(start), 'end': stamp(end),\r\n            'durationMinutes': float(end - start), 'raw': raw if raw is not None else {'id': rid}}\r\n\r\n\r\ndef sap(rid, start, end, machine='A', sub='FORNO 1', text='TROCA BCL', **extra):\r\n    row = {'id': rid, 'machine': machine, 'subdivision': sub, 'text': text, 'start': stamp(start),\r\n           'end': stamp(end) if end is not None else None,\r\n           'durationMinutes': float(end - start) if end is not None else None,\r\n           'mappingStatus': 'inferido de segmento MAQ; validar tecnicamente', 'raw': {'id': rid}}\r\n    row.update(extra)\r\n    return row\r\n\r\n\r\ndef decisions(result):\r\n    return {d['id']: d for d in result['decisions']}\r\n\r\n\r\nclass SyntheticCases(unittest.TestCase):\r\n    def test_unique_pair_is_probable_but_pending(self):\r\n        r = ce.analyze([mes('M1', 0, 60)], [sap('S1', 5, 55)])\r\n        d = decisions(r)\r\n        self.assertEqual(d['M1']['decisao'], 'correspondencia_provavel')\r\n        self.assertEqual(d['S1']['decisao'], 'correspondencia_provavel')\r\n        self.assertEqual(d['M1']['contraparteId'], 'S1')\r\n        self.assertEqual(d['S1']['revisao'], 'pendente')\r\n        self.assertEqual(d['S1']['cadastro'], 'inferido')\r\n        self.assertIn('cadastro_pendente', d['S1']['motivosConfianca'])\r\n        self.assertEqual(r['pairs'][0]['classe'], 'forte')\r\n\r\n    def test_dispute_has_no_automatic_tiebreak(self):\r\n        r = ce.analyze([mes('M1', 0, 60)], [sap('S1', 5, 55), sap('S2', 10, 50)])\r\n        d = decisions(r)\r\n        self.assertEqual({d[i]['decisao'] for i in ('M1', 'S1', 'S2')}, {'disputa'})\r\n        self.assertIsNone(d['M1']['contraparteId'])\r\n        self.assertEqual(d['M1']['nForte'], 2)\r\n\r\n    def test_absence_on_each_side(self):\r\n        r = ce.analyze([mes('M1', 0, 30), mes('M2', 1000, 1030)], [sap('S1', 5, 25), sap('S2', 5000, 5030)])\r\n        d = decisions(r)\r\n        self.assertEqual(d['M2']['decisao'], 'sem_candidato')\r\n        self.assertIn('mes_sem_candidato', d['M2']['saidas'])\r\n        self.assertEqual(d['S2']['decisao'], 'sem_candidato')\r\n        self.assertIn('sap_sem_candidato', d['S2']['saidas'])\r\n        self.assertIsNone(d['M2']['classeMaxima'])\r\n        self.assertEqual(d['M2']['cadastro'], 'nao_aplicavel')\r\n\r\n    def test_different_machine_is_never_candidate(self):\r\n        r = ce.analyze([mes('M1', 0, 60, machine='A')], [sap('S1', 0, 60, machine='B')])\r\n        self.assertEqual(r['pairs'], [])\r\n\r\n    def test_exact_duplicate_and_possible_duplicate_are_distinct_flags(self):\r\n        raw = {'x': 1}\r\n        records = [mes('M1', 0, 30, raw=raw), mes('M2', 0, 30, raw=raw)]\r\n        flags = ce.analyze(records, [])['flags']\r\n        self.assertEqual([f['tipo'] for f in flags], ['duplicata_exata'])\r\n        records = [mes('M1', 0, 30), mes('M2', 10, 40)]\r\n        flags = ce.analyze(records, [])['flags']\r\n        self.assertEqual([f['tipo'] for f in flags], ['possivel_duplicidade'])\r\n        self.assertIn('possivel_duplicidade', decisions(ce.analyze(records, []))['M1']['saidas'])\r\n        self.assertEqual(len(records), 2)  # nada é apagado\r\n\r\n    def test_same_text_or_overlap_alone_is_not_duplicate(self):\r\n        self.assertEqual(ce.analyze([mes('M1', 0, 30, sub='FORNO 1'), mes('M2', 10, 40, sub='DISCO')], [])['flags'], [])\r\n        self.assertEqual(ce.analyze([mes('M1', 0, 30), mes('M2', 100, 130)], [])['flags'], [])\r\n\r\n    def test_simultaneous_sap_with_different_texts_get_no_flag(self):\r\n        r = ce.analyze([mes('M1', 0, 60)], [sap('S1', 0, 60, text='TROCA BCL'), sap('S2', 0, 60, text='LIMPEZA GERAL')])\r\n        self.assertEqual(r['flags'], [])\r\n\r\n    def test_fragmentation_flagged_not_merged(self):\r\n        r = ce.analyze([mes('M1', 0, 30), mes('M2', 31, 60)], [sap('S1', 0, 60)])\r\n        self.assertEqual([f['tipo'] for f in r['flags']], ['possivel_fragmentacao'])\r\n        self.assertEqual(r['flags'][0]['campos']['gapMin'], 1.0)\r\n        self.assertEqual(len([d for d in r['decisions'] if d['sistema'] == 'MES']), 2)\r\n        ids = {tuple(h['mesIds']) for h in r['hypotheses']}\r\n        self.assertIn(('M1',), ids)\r\n        self.assertIn(('M2',), ids)\r\n        gap_over = ce.analyze([mes('M1', 0, 30), mes('M2', 31.5, 60)], [])\r\n        self.assertEqual(gap_over['flags'], [])\r\n\r\n    def test_n_to_one_hypothesis_uses_union_and_keeps_gap(self):\r\n        r = ce.analyze([mes('M1', 0, 30), mes('M2', 50, 80)], [sap('S1', 0, 70)])\r\n        h = [h for h in r['hypotheses'] if h['tipo'] == 'N:1'][0]\r\n        self.assertEqual(h['mesIds'], ['M1', 'M2'])\r\n        self.assertEqual(h['uniao']['MES']['duracaoMin'], 60.0)\r\n        self.assertEqual(h['uniao']['MES']['lacunas'][0]['min'], 20.0)\r\n        self.assertEqual(h['uniao']['SAP']['duracaoMin'], 70.0)\r\n        self.assertTrue(h['duracao']['relevante'] is False)\r\n        self.assertEqual(h['estado'], 'proposta')\r\n        self.assertIn('1:1', {x['tipo'] for x in r['hypotheses']})  # alternativas concorrentes preservadas\r\n\r\n    def test_one_to_n_simultaneous_legitimate_without_duplicate_flag(self):\r\n        r = ce.analyze([mes('M1', 0, 120)], [sap('S1', 0, 60, text='TROCA BCL'), sap('S2', 60, 120, text='AJUSTE SENSOR BCL')])\r\n        self.assertIn('1:N', {h['tipo'] for h in r['hypotheses']})\r\n        self.assertEqual(r['flags'], [])\r\n\r\n    def test_n_to_m_block_and_limit(self):\r\n        mes_rows = [mes('M1', 0, 40), mes('M2', 20, 60)]\r\n        sap_rows = [sap('S1', 0, 40), sap('S2', 20, 60)]\r\n        r = ce.analyze(mes_rows, sap_rows)\r\n        self.assertIn('N:M', {h['tipo'] for h in r['hypotheses']})\r\n        self.assertEqual(len(r['blocks']), 1)\r\n        self.assertIn('não é evento confirmado', r['blocks'][0]['rotulo'])\r\n        big = ce.analyze([mes(f'M{i}', 0, 100) for i in range(7)], [sap(f'S{i}', 0, 100) for i in range(7)])\r\n        nm = [h for h in big['hypotheses'] if h['tipo'] == 'N:M'][0]\r\n        self.assertEqual(nm['estado'], 'nm_nao_enumerado')\r\n\r\n    def test_incomplete_sap_interval(self):\r\n        r = ce.analyze([mes('M1', 0, 60)], [sap('S1', 10, None)])\r\n        p = r['pairs'][0]\r\n        self.assertEqual((p['temporal'], p['classe'], p['sapSemFim']), ('sap_sem_fim', 'fraca', True))\r\n        d = decisions(r)\r\n        self.assertEqual(d['M1']['decisao'], 'evidencia_insuficiente')\r\n        self.assertTrue(d['S1']['intervaloIncompleto'])\r\n        self.assertIn('intervalo_incompleto', d['M1']['motivosConfianca'])\r\n        h = [h for h in r['hypotheses'] if h['tipo'] == '1:1']\r\n        self.assertEqual(h, [])  # fraca não gera hipótese\r\n        self.assertIsNone(p['duracao']['relevante'])\r\n        self.assertFalse(p['duracao']['comparavel'])\r\n\r\n    def test_long_sap_window_preserved_and_flagged(self):\r\n        r = ce.analyze([mes('M1', 0, 60)], [sap('S1', -10, 60 * 30)])\r\n        p = r['pairs'][0]\r\n        self.assertTrue(p['janelaSapLonga'])\r\n        h = r['hypotheses'][0]\r\n        self.assertEqual(h['janelaSapLonga'], ['S1'])\r\n        self.assertEqual(h['uniao']['SAP']['duracaoMin'], 60 * 30 + 10)\r\n        self.assertEqual(r['summary']['sapJanelaLonga'], 1)\r\n\r\n    def test_legitimately_different_duration_flagged_without_downgrade(self):\r\n        r = ce.analyze([mes('M1', 0, 60)], [sap('S1', 20, 40)])\r\n        p = r['pairs'][0]\r\n        self.assertEqual(p['classe'], 'forte')\r\n        self.assertTrue(p['duracao']['relevante'])\r\n        self.assertEqual(p['duracao']['absolutaMin'], 40.0)\r\n        self.assertIn('divergencia_duracao', decisions(r)['M1']['saidas'])\r\n        self.assertEqual(decisions(r)['M1']['decisao'], 'correspondencia_provavel')\r\n\r\n    def test_both_zero_durations_by_convention(self):\r\n        self.assertEqual(ce.duration_divergence(0.0, 0.0)['razao'], 0.0)\r\n        self.assertFalse(ce.duration_divergence(0.0, 0.0)['relevante'])\r\n        r = ce.analyze([mes('M1', 10, 10)], [sap('S1', 10, 10)])\r\n        self.assertFalse(r['pairs'][0]['duracao']['relevante'])\r\n        self.assertEqual(r['hypotheses'][0]['duracao']['motivo'], 'ambas_duracoes_zero_convencao_razao_zero')\r\n\r\n    def test_duration_threshold_is_both_absolute_and_relative(self):\r\n        self.assertFalse(ce.duration_divergence(100, 114)['relevante'])      # <15 min\r\n        self.assertFalse(ce.duration_divergence(100, 115)['relevante'])      # 13%\r\n        self.assertTrue(ce.duration_divergence(40, 60)['relevante'])         # 20 min e 33%\r\n        self.assertIsNone(ce.duration_divergence(None, 5)['relevante'])\r\n\r\n    def test_different_subdivision_is_unknown_not_incompatible(self):\r\n        r = ce.analyze([mes('M1', 0, 60, sub='AQUECIMENTO', comp='')], [sap('S1', 0, 60, sub='FORNO 1', text='SEM CITACAO')])\r\n        p = r['pairs'][0]\r\n        self.assertEqual(p['subdivisao'], 'diferente_compatibilidade_desconhecida')\r\n        self.assertEqual(p['classe'], 'moderada')\r\n        self.assertNotIn('incompativel', json.dumps(r))\r\n\r\n    def test_absent_information_is_not_contradiction(self):\r\n        r = ce.analyze([mes('M1', 0, 60, sub='', comp='')], [sap('S1', 0, 60, sub='', text='')])\r\n        p = r['pairs'][0]\r\n        self.assertEqual((p['subdivisao'], p['componente'], p['classe']), ('ausente', 'ausente', 'moderada'))\r\n\r\n    def test_proximity_classes(self):\r\n        near = ce.analyze([mes('M1', 0, 60)], [sap('S1', 70, 100)])['pairs'][0]\r\n        self.assertEqual((near['temporal'], near['classe']), ('proximidade_curta', 'moderada'))\r\n        far = ce.analyze([mes('M1', 0, 60)], [sap('S1', 100, 130)])['pairs'][0]\r\n        self.assertEqual((far['temporal'], far['classe']), ('proximidade', 'moderada'))\r\n        touch = ce.analyze([mes('M1', 0, 60)], [sap('S1', 60, 90)])['pairs'][0]\r\n        self.assertEqual((touch['temporal'], touch['classe']), ('contato', 'forte'))\r\n\r\n    def test_decisions_symmetric_and_complete(self):\r\n        r = ce.analyze([mes('M1', 0, 60), mes('M2', 200, 260)], [sap('S1', 5, 55), sap('S2', 205, 255), sap('S3', 205, 255)])\r\n        d = decisions(r)\r\n        self.assertEqual(sorted(d), ['M1', 'M2', 'S1', 'S2', 'S3'])\r\n        for item in r['decisions']:\r\n            if item['decisao'] == 'correspondencia_provavel':\r\n                self.assertEqual(d[item['contraparteId']]['contraparteId'], item['id'])\r\n\r\n    def test_duplicate_ids_rejected(self):\r\n        with self.assertRaises(ValueError):\r\n            ce.analyze([mes('X', 0, 10)], [sap('X', 0, 10)])\r\n\r\n\r\nclass Equivalence(unittest.TestCase):\r\n    def scenario(self):\r\n        mes_rows = [mes('M1', 0, 60), mes('M2', 500, 560), mes('M3', 1000, 1000)]\r\n        sap_rows = [\r\n            sap('S_gap60', 120, 150), sap('S_gap60_001', 120.001 - 0.0, 150),\r\n            sap('S_before60', -90, -60), sap('S_before60_001', -90, -60.001),\r\n            sap('S_open_in', 119, None), sap('S_open_out', 121, None), sap('S_open_before', -60, None), sap('S_open_before2', -61, None),\r\n            sap('S_far30', 90, 100), sap('S_far120', 170, 200), sap('S_long', -3000, 520), sap('S_other', 0, 60, machine='B'),\r\n        ]\r\n        return mes_rows, sap_rows\r\n\r\n    def test_indexed_equals_bruteforce_and_v3_at_boundaries(self):\r\n        mes_rows, sap_rows = self.scenario()\r\n        for window in (30, 60, 120):\r\n            self.assertEqual(set(ce.indexed_pairs(mes_rows, sap_rows, window)), set(ce.brute_force_pairs(mes_rows, sap_rows, window)), window)\r\n        original = v3.POLICY['candidateWindowMinutes']\r\n        try:\r\n            for window in (30, 60, 120):\r\n                v3.POLICY['candidateWindowMinutes'] = window\r\n                v3_pairs = {(c['mesId'], c['sapId']) for c in v3.associate(mes_rows, sap_rows)['candidates']}\r\n                self.assertEqual(set(ce.indexed_pairs(mes_rows, sap_rows, window)), v3_pairs, window)\r\n        finally:\r\n            v3.POLICY['candidateWindowMinutes'] = original\r\n        names = {s for _, s in ce.indexed_pairs(mes_rows, sap_rows, 60)}\r\n        self.assertIn('S_gap60', names)\r\n        self.assertNotIn('S_gap60_001', names)\r\n        self.assertIn('S_before60', names)\r\n        self.assertNotIn('S_before60_001', names)\r\n        self.assertIn('S_open_in', names)\r\n        self.assertNotIn('S_open_before2', names)\r\n\r\n    def test_long_sap_found_by_index(self):\r\n        r = ce.indexed_pairs([mes('M1', 1000, 1060)], [sap('S1', 0, 5000)], 60)\r\n        self.assertEqual(r, [('M1', 'S1')])\r\n\r\n\r\nclass Determinism(unittest.TestCase):\r\n    def rows(self):\r\n        return ([mes('M1', 0, 60), mes('M2', 30, 90), mes('M3', 400, 430)],\r\n                [sap('S1', 0, 60), sap('S2', 40, 100), sap('S3', 405, None), sap('S4', 9000, 9100)])\r\n\r\n    def test_two_runs_identical_hash(self):\r\n        a = ce.analyze(*self.rows())\r\n        b = ce.analyze(*copy.deepcopy(self.rows()))\r\n        self.assertEqual(ce.content_hash(a), ce.content_hash(b))\r\n\r\n    def test_every_id_in_exactly_one_decision(self):\r\n        mes_rows, sap_rows = self.rows()\r\n        ids = [d['id'] for d in ce.analyze(mes_rows, sap_rows)['decisions']]\r\n        self.assertEqual(sorted(ids), sorted(r['id'] for r in mes_rows + sap_rows))\r\n        self.assertEqual(len(ids), len(set(ids)))\r\n\r\n    def test_input_not_mutated(self):\r\n        rows = self.rows()\r\n        before = copy.deepcopy(rows)\r\n        ce.analyze(*rows)\r\n        self.assertEqual(rows, before)\r\n\r\n    def test_union_intervals_never_uses_amplitude(self):\r\n        u = ce.union_intervals([(BASE, BASE + timedelta(minutes=10)), (BASE + timedelta(minutes=100), BASE + timedelta(minutes=110)),\r\n                                (BASE + timedelta(minutes=5), BASE + timedelta(minutes=20))])\r\n        self.assertEqual(u['duracaoMin'], 30.0)\r\n        self.assertEqual(len(u['lacunas']), 1)\r\n        self.assertEqual(u['lacunas'][0]['min'], 80.0)\r\n\r\n\r\nclass Database(unittest.TestCase):\r\n    def test_temp_database_one_decision_per_record_and_fks(self):\r\n        from gerar_snapshot_evidencias import build_snapshot, write_database, history_like\r\n        mes_rows = [mes('MES-R0002', 0, 60), mes('MES-R0003', 500, 560)]\r\n        sap_rows = [sap('SAP-R0002', 5, 55), sap('SAP-R0003', 900, 950)]\r\n        for i, row in enumerate(mes_rows + sap_rows):\r\n            row['source'] = {'file': 'f.xlsx', 'sheet': 'S', 'row': i + 2, 'hash': 'a' * 64}\r\n        history = history_like(mes_rows, sap_rows)\r\n        snapshot = build_snapshot(history, input_sha256='b' * 64)\r\n        with tempfile.TemporaryDirectory() as tmp:\r\n            path = Path(tmp) / 'ev.sqlite'\r\n            summary = write_database(history, snapshot, path)\r\n            self.assertEqual(summary['integrity'], 'ok')\r\n            self.assertEqual(summary['foreignKeyErrors'], 0)\r\n            db = sqlite3.connect(path)\r\n            self.assertEqual(db.execute('select count(*), count(distinct record_id) from ev_record_decision').fetchone(), (4, 4))\r\n            self.assertEqual(db.execute('select count(*) from ev_review').fetchone()[0], 0)\r\n            self.assertEqual(db.execute('select count(*) from source_record').fetchone()[0], 4)\r\n            with self.assertRaises(FileExistsError):\r\n                write_database(history, snapshot, path)\r\n            db.close()\r\n\r\n\r\nif __name__ == '__main__':\r\n    unittest.main()\r\n",
 "05_subprodutos/catalogo_ativos.json": "{\n  \"schemaVersion\": 1,\n  \"assets\": [\n    { \"id\": \"A\", \"sourceKey\": \"A\", \"displayName\": \"Equipamento A\", \"assetNature\": \"real anonymized\", \"studyFocus\": true },\n    { \"id\": \"B\", \"sourceKey\": \"B\", \"displayName\": \"Equipamento B\", \"assetNature\": \"fictitious\", \"studyFocus\": false },\n    { \"id\": \"C\", \"sourceKey\": \"C\", \"displayName\": \"Equipamento C\", \"assetNature\": \"fictitious\", \"studyFocus\": false },\n    { \"id\": \"D\", \"sourceKey\": \"D\", \"displayName\": \"Equipamento D\", \"assetNature\": \"fictitious\", \"studyFocus\": false }\n  ]\n}\r\n"
}
SOURCE_HASHES = {
 "04_dados/auditoria/gpt/GPT-DADOS-01/auditar_fontes.py": "a80dddfde4660b7d28af762bd21cdbc5c78375f9badf7f739145f710c83d9f08",
 "05_subprodutos/gerar_historico_mvp.py": "9cb18b141b786226d4f2a15212375763ce185b4de70c9438976d7aaa2d99a814",
 "05_subprodutos/02_conciliacao/associacao_automatica.py": "be948283ea3eb92b8c699a5bd23e876af362e45d82f2650d33dc2be7ec720d6e",
 "05_subprodutos/02_conciliacao/conciliacao_evidencias.py": "14abdc9dc4bab06d4875bf56b4acf7e9d667e84bc893d2c21e9c56bf880a6ced",
 "05_subprodutos/02_conciliacao/gerar_snapshot_evidencias.py": "458418ec200fed63485e9203d0fda5fd870c9c95476ab1caa27b97b0506c96b3",
 "05_subprodutos/02_conciliacao/criar_base_consolidada.py": "15796c48fb25de3aec8fa08399cd21774893c7cda993f8a19686f48aef3d4b17",
 "05_subprodutos/02_conciliacao/test_associacao_automatica.py": "71375828727a4a79ecc7c870b88798f0e59303d653b5a89cb3648acaffd283ae",
 "05_subprodutos/02_conciliacao/test_conciliacao_evidencias.py": "08b14b08195c6196462d4ad52d3d26664717956037fe7ce9c1f83e5aada23089",
 "05_subprodutos/catalogo_ativos.json": "ae78043a9fb57f160b06368275f6e15ee3737b447066bdd8be1c7394e0db6f43"
}

In [ ]:
def gravar_modulos(destino):
    """Grava os arquivos embutidos e confere o SHA-256 de cada um."""
    for relativo, codigo in SOURCE_FILES.items():
        caminho = destino / relativo
        caminho.parent.mkdir(parents=True, exist_ok=True)
        caminho.write_bytes(codigo.encode('utf8'))
        obtido = hashlib.sha256(caminho.read_bytes()).hexdigest()
        assert obtido == SOURCE_HASHES[relativo], f'Hash divergente: {relativo}'


def carregar_modulo(nome, caminho):
    spec = importlib.util.spec_from_file_location(nome, caminho)
    modulo = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(modulo)
    return modulo


gravar_modulos(PASTA)
CONC = PASTA / '05_subprodutos/02_conciliacao'
if str(CONC) not in sys.path:
    sys.path.insert(0, str(CONC))
ASSOCIACAO = carregar_modulo('sime_associacao', CONC / 'associacao_automatica.py')
BASE = carregar_modulo('sime_base', CONC / 'criar_base_consolidada.py')
import conciliacao_evidencias as EVIDENCIAS_MOD
import gerar_snapshot_evidencias as GERADOR_EVIDENCIAS
display(pd.DataFrame([{'arquivo': p, 'SHA-256': h} for p, h in SOURCE_HASHES.items()]))

## 3. Entrada <a id="entrada" name="entrada"></a>

- `demo` (padrão): planilha fictícia com 3 paradas MES e 2 ordens SAP.
- `upload`: XLSX de até 25 MiB com as abas **MES - PRODUÇÃO** e **SAP - MANUTENÇÃO** e os cabeçalhos do case.
- `local`: copia o arquivo de `SIME_INPUT_XLSX`.

No Colab, a planilha é processada pelo Google; envie só dados autorizados.

In [ ]:
MODO = os.environ.get('SIME_MODO', 'demo')  # no Colab, troque para 'upload'

# Cabeçalhos exatos do case (inclusive o espaço em 'MÁQUINA ').
MES_COLS = ['MÁQUINA ', 'Data', 'Famila', 'EQUIPAMENTO', 'Componente', 'Tur', 'Inicio', 'Término', 'Horas']
SAP_COLS = ['N. Ordem', 'Tipo', 'Data-base iníc.', 'Texto breve', 'Loc.instalação', 'Descrição',
            'Dta.início real', 'InícReal hr.', 'Dt.real fim', 'Fim real', 'Tip.ativ.manut.']


def criar_planilha_demo(destino):
    """Planilha fictícia: três paradas de 30 min na máquina A e duas ordens SAP."""
    wb = openpyxl.Workbook()
    wb.remove(wb.active)
    mes = wb.create_sheet('MES - PRODUÇÃO')
    sap = wb.create_sheet('SAP - MANUTENÇÃO')
    mes.append(MES_COLS)
    sap.append(SAP_COLS)
    dia = datetime(2026, 1, 2)
    for subdivisao, componente, hora in [('LONAS', 'LONA', 10), ('DISCO', 'QMD', 12), ('LONAS', 'LONA', 15)]:
        inicio = dia + timedelta(hours=hora)
        mes.append(['A', dia, 'QUEBRA', subdivisao, componente, 1, inicio, inicio + timedelta(minutes=30), 0.5])
    for ordem, subdivisao, texto, hora in [(100001, 'LONAS', 'LONA TRAVADA', 10), (100002, 'DISCO', 'VERIFICAR QMD', 12)]:
        sap.append([ordem, 'Corretiva', dia, texto, '9001.FAB.CEL.MAQA.X', subdivisao,
                    dia, time(hora, 0), dia, time(hora, 30), 'CO'])
    wb.save(destino)
    return destino


def receber_upload(pasta):
    if not EM_COLAB:
        raise RuntimeError('Upload interativo requer Colab. Use demo ou local.')
    from google.colab import files
    enviados = files.upload()
    if len(enviados) != 1:
        raise ValueError('Selecione uma única planilha XLSX.')
    nome, conteudo = next(iter(enviados.items()))
    if Path(nome).suffix.lower() != '.xlsx' or len(conteudo) > 25 * 1024 * 1024:
        raise ValueError('Envie um XLSX de até 25 MiB.')
    destino = pasta / Path(nome).name
    destino.write_bytes(conteudo)
    return destino


def copiar_local(pasta):
    original = Path(os.environ['SIME_INPUT_XLSX'])
    destino = pasta / original.name
    if original.resolve() != destino.resolve():
        shutil.copyfile(original, destino)
    return destino


if MODO == 'demo':
    ARQUIVO = criar_planilha_demo(PASTA / 'DEMO_SINTETICA_MES_SAP.xlsx')
elif MODO == 'upload':
    ARQUIVO = receber_upload(PASTA)
elif MODO == 'local':
    ARQUIVO = copiar_local(PASTA)
else:
    raise ValueError('MODO deve ser demo, upload ou local.')
print('Modo:', MODO, '| arquivo:', ARQUIVO.name, '| bytes:', ARQUIVO.stat().st_size)

## 4. Validação <a id="validacao" name="validacao"></a>

São recusados macros, links externos, ZIP grande demais, abas ou colunas faltando, cabeçalho duplicado e tabela vazia. Fórmulas não são recalculadas. O auditor do projeto gera `MES.csv`, `SAP.csv` e um perfil de qualidade.

In [ ]:
MAX_ENTRADAS_ZIP = 5000
MAX_BYTES_EXPANDIDOS = 100 * 1024 * 1024
MAX_LINHAS, MAX_COLUNAS = 100_000, 50


def checar_pacote_xlsx(caminho):
    """Um .xlsx é um ZIP: limita tamanho expandido e recusa macros e links externos."""
    with zipfile.ZipFile(caminho) as z:
        entradas = z.infolist()
    if len(entradas) > MAX_ENTRADAS_ZIP or sum(i.file_size for i in entradas) > MAX_BYTES_EXPANDIDOS:
        raise ValueError('Planilha excede o limite de expansão.')
    if any('vbaProject' in i.filename or 'externalLinks/' in i.filename for i in entradas):
        raise ValueError('Macros/relações externas não são aceitas.')


def checar_abas(caminho):
    """Confere abas, cabeçalhos e tamanho das tabelas sem executar fórmulas."""
    wb = openpyxl.load_workbook(caminho, read_only=True, data_only=True)
    try:
        for aba, obrigatorias in [('MES - PRODUÇÃO', MES_COLS), ('SAP - MANUTENÇÃO', SAP_COLS)]:
            if aba not in wb.sheetnames:
                raise ValueError('Aba ausente: ' + aba)
            planilha = wb[aba]
            cabecalho = list(next(planilha.iter_rows(values_only=True)))
            if len(cabecalho) != len(set(cabecalho)):
                raise ValueError('Cabeçalhos duplicados: ' + aba)
            faltando = set(obrigatorias) - set(cabecalho)
            if faltando:
                raise ValueError(f'Colunas ausentes em {aba}: {sorted(faltando)}')
            if planilha.max_row < 2 or planilha.max_row > MAX_LINHAS or planilha.max_column > MAX_COLUNAS:
                raise ValueError('Tamanho de tabela fora do limite.')
    finally:
        wb.close()


def registrar_fonte(caminho):
    """Inventário com o hash da planilha, usado pela normalização para rastrear a origem."""
    inventario = {'fontes': [{'arquivo': caminho.name, 'bytes': caminho.stat().st_size,
                              'sha256': hashlib.sha256(caminho.read_bytes()).hexdigest()}]}
    (PASTA / '01_fontes').mkdir(exist_ok=True)
    (PASTA / '01_fontes/inventario.json').write_text(json.dumps(inventario, ensure_ascii=False), encoding='utf8')


checar_pacote_xlsx(ARQUIVO)
checar_abas(ARQUIVO)
AUDITOR = carregar_modulo('sime_auditoria', PASTA / '04_dados/auditoria/gpt/GPT-DADOS-01/auditar_fontes.py')
AUDITOR.FILE = ARQUIVO
AUDITOR.main()
registrar_fonte(ARQUIVO)

resumo = json.loads((AUDITOR.OUT / 'resumo.json').read_text(encoding='utf8'))
display(pd.DataFrame([{'sistema': k, 'registros': v} for k, v in resumo['registros'].items()]))
display(resumo['limites'])

## 5. Associação <a id="associacao" name="associacao"></a>

As datas viram ISO sem alterar os campos originais. O notebook mantém a v3 como comparação e executa em paralelo `sp2-evidence-v1`, o novo método proposto.

### V3 — referência de comparação

Compara registros do mesmo ativo numa janela de 60 min e soma sinais de subdivisão, componente, intervalo e duração. O score só ordena candidatos; não é probabilidade. A v3 pode sugerir quando o ativo SAP foi inferido. A decisão nova não substitui nem apaga o resultado v3.

| Evidência | Pontos |
|---|---|
| Mesmo ativo (obrigatório) | 20 |
| Subdivisão idêntica | 25 |
| Componente MES (≥ 3 letras) no texto SAP | 15 |
| Sobreposição / proximidade / ordem sem fim a ≤ 15 min | 12 / 10 / 5 |
| Início a ≤ 5 / 15 / 60 min | 10 / 7 / 3 |
| Fim a ≤ 5 / 15 / 60 min | 10 / 7 / 3 |
| Duração parecida | 15 / 10 / 5 |

O score (máx. 100) só ordena candidatos; não é probabilidade. A regra automática exige score ≥ 90, vantagem ≥ 10 sobre o segundo nos dois sentidos, sem empate, ordem SAP com fim e subdivisão ou componente coincidente.

### `sp2-evidence-v1` — método candidato

1. Indexa registros por máquina e tempo, reduzindo comparações improváveis. Mantém a mesma janela de candidatos da v3 (60 min, fronteira inclusiva).
2. Para cada par, expõe geometria temporal, subdivisão, componente citado, estado do cadastro SAP e diferença de duração. Subdivisão diferente significa compatibilidade desconhecida, pois não há mapa técnico validado.
3. Classifica evidência como **forte, moderada ou fraca** (classes ordinais, sem percentual). Uma correspondência provável requer único candidato forte recíproco; concorrência forte vira disputa; candidatos sem sinal forte ficam como evidência insuficiente; ausência de candidatos fica separada.
4. Registra uma decisão para cada ID MES **e** SAP, com motivos, alternativas, cadastro inferido e revisão pendente. Não promove vínculo confirmado automaticamente.
5. Agrupa candidatos em blocos de investigação e propõe hipóteses 1:1, N:1, 1:N e N:M. Bloco/hipótese organiza revisão, não confirma um evento físico nem une fragmentos.
6. Sinaliza possíveis duplicidades/fragmentação sem excluir ou unir linhas; compara durações pela união dos intervalos de cada fonte, preservando lacunas.

Os limiares de 60 min, 15 min para proximidade curta, 1 min para possível fragmentação e 15 min + 25% para triagem de duração são **propostas**, ainda não calibradas por revisão humana. Compare 30/60/120 min e revise casos; cobertura não é acurácia.

In [ ]:
CATALOGO = PASTA / '05_subprodutos/catalogo_ativos.json'
if MODO == 'demo':  # marca os ativos como fictícios para não confundir com o case real
    catalogo = json.loads(CATALOGO.read_text(encoding='utf8'))
    for ativo in catalogo['assets']:
        ativo['assetNature'] = 'fictício'
    CATALOGO.write_text(json.dumps(catalogo, ensure_ascii=False), encoding='utf8')

# Normaliza MES/SAP e roda a associação com a política padrão.
HISTORICO = PASTA / 'saidas/historico.json'
HISTORICO.parent.mkdir(exist_ok=True)
subprocess.run([sys.executable, str(PASTA / '05_subprodutos/gerar_historico_mvp.py'),
                '--output', str(HISTORICO), '--report', str(PASTA / 'saidas/geracao.json')], check=True)
DADOS = json.loads(HISTORICO.read_text(encoding='utf8'))
associacao = DADOS['association']

display(pd.Series(ASSOCIACAO.POLICY, name='valor').to_frame('política'))
display(pd.DataFrame([{'status': k, 'quantidade': v} for k, v in associacao['summary'].items()]))
print('MES:', len(DADOS['mes']), '| SAP:', len(DADOS['sap']), '| candidatos:', len(associacao['candidates']))
print('Vínculos selecionados:', len(associacao['automaticLinks']), '| sugestões:', len(associacao['automaticSuggestions']))

# Método candidato em paralelo: preserva o JSON da v3 e gera snapshots/saídas com nomes novos.
SUFIXO = uuid4().hex[:8]
SNAPSHOT_EVIDENCIAS = PASTA / 'saidas' / f'sp2-evidencias-{SUFIXO}.json'
BANCO_EVIDENCIAS = PASTA / 'saidas' / f'sp2-evidencias-{SUFIXO}.sqlite'
HISTORICO_ADITIVO = PASTA / 'saidas' / f'historico-com-evidencias-{SUFIXO}.json'
GERACAO_EVIDENCIAS_PROC = subprocess.run([
    sys.executable, str(CONC / 'gerar_snapshot_evidencias.py'), '--input', str(HISTORICO),
    '--snapshot-output', str(SNAPSHOT_EVIDENCIAS), '--db-output', str(BANCO_EVIDENCIAS),
    '--history-output', str(HISTORICO_ADITIVO), '--window', '60'
], capture_output=True, text=True, encoding='utf8', errors='replace')
print(GERACAO_EVIDENCIAS_PROC.stdout)
if GERACAO_EVIDENCIAS_PROC.stderr:
    print(GERACAO_EVIDENCIAS_PROC.stderr)
GERACAO_EVIDENCIAS_PROC.check_returncode()
GERACAO_EVIDENCIAS = json.loads(GERACAO_EVIDENCIAS_PROC.stdout)
EVIDENCIAS = json.loads(SNAPSHOT_EVIDENCIAS.read_text(encoding='utf8'))
RESULTADO_EVIDENCIAS = EVIDENCIAS['result']
display(pd.DataFrame([
    {'saída §5.2': nome, 'registros sinalizados': quantidade}
    for nome, quantidade in RESULTADO_EVIDENCIAS['summary']['saidas'].items()
]))
display(pd.DataFrame([
    {'sistema': sistema, 'decisão': decisao, 'registros': quantidade}
    for sistema, decisoes in RESULTADO_EVIDENCIAS['summary']['decisoes'].items()
    for decisao, quantidade in decisoes.items()
]))
print('Método:', EVIDENCIAS['rulesVersion'], '| janela:', EVIDENCIAS['window'], 'minutos')
print('Decisões:', len(RESULTADO_EVIDENCIAS['decisions']), '| pares com sinais:', len(RESULTADO_EVIDENCIAS['pairs']),
      '| blocos:', len(RESULTADO_EVIDENCIAS['blocks']), '| hipóteses:', len(RESULTADO_EVIDENCIAS['hypotheses']),
      '| flags:', len(RESULTADO_EVIDENCIAS['flags']), '| revisões humanas:', len(EVIDENCIAS['reviews']))

# Sensibilidade proposta: compare cobertura/decisões, sem interpretar estabilidade como acurácia.
linhas_sensibilidade = []
for janela_teste in (30, 60, 120):
    resumo_teste = EVIDENCIAS_MOD.analyze_history(DADOS, window=janela_teste)['summary']
    for sistema, decisoes in resumo_teste['decisoes'].items():
        linhas_sensibilidade.append({'janela_min': janela_teste, 'sistema': sistema,
            'provavel': decisoes.get('correspondencia_provavel', 0),
            'disputa': decisoes.get('disputa', 0),
            'evidencia_insuficiente': decisoes.get('evidencia_insuficiente', 0),
            'sem_candidato': decisoes.get('sem_candidato', 0)})
display(pd.DataFrame(linhas_sensibilidade))

## 6. Banco SQLite <a id="banco" name="banco"></a>

As linhas não são fundidas. O banco v3 da seção abaixo mantém o baseline; o banco de evidências usa o mesmo esquema de origem/candidatos e acrescenta tabelas `ev_*`: sinais por par, uma decisão por MES e SAP, blocos, hipóteses, flags e revisões (vazia até uma análise humana).

Cada execução cria bancos novos. A geração do método candidato confere integridade SQLite, chaves estrangeiras e cobertura das decisões. A base aditiva `HISTORICO_ADITIVO` mantém a v3 e acrescenta `associationCandidate`.

In [ ]:
def verificar_banco(caminho, total_esperado):
    with sqlite3.connect(caminho) as con:
        con.execute('PRAGMA foreign_keys=ON')
        assert con.execute('PRAGMA integrity_check').fetchone()[0] == 'ok'
        assert not con.execute('PRAGMA foreign_key_check').fetchall()
        total = con.execute('SELECT COUNT(*) FROM source_record').fetchone()[0]
        assert total == total_esperado, (total, total_esperado)


BANCO = PASTA / 'saidas' / f'sime_mes_sap_{uuid4().hex[:8]}.sqlite'
VALIDACAO = BASE.create_database(HISTORICO, BANCO)
verificar_banco(BANCO, len(DADOS['mes']) + len(DADOS['sap']))
display(VALIDACAO)
print('Banco:', BANCO.name)

## 7. Consultas <a id="consultas" name="consultas"></a>

Troque `ID_MES` para ver todos os candidatos de uma parada. As consultas usam parâmetros (`?`), sem concatenar texto no SQL.

In [ ]:
def consultar(sql, params=()):
    with sqlite3.connect(BANCO) as con:
        return pd.read_sql_query(sql, con, params=params)


display(consultar('SELECT association_status, COUNT(*) AS quantidade FROM association_joined GROUP BY association_status'))
display(consultar('SELECT mes_id, asset_id, association_status, association_score, selected_sap_id, suggested_sap_id '
                  'FROM association_joined LIMIT 12'))

In [ ]:
ID_MES = DADOS['mes'][0]['id']  # troque por um ID da tabela acima

display(consultar('SELECT * FROM association_decision WHERE mes_id = ?', (ID_MES,)))
display(consultar('SELECT sap_id, score, evidence_json FROM association_candidate '
                  'WHERE mes_id = ? ORDER BY score DESC, sap_id', (ID_MES,)))

In [ ]:
ID_DATASET = GERACAO_EVIDENCIAS['database']['datasetId']
VERSAO_REGRAS = EVIDENCIAS['rulesVersion']

def consultar_evidencias(sql, params=()):
    with sqlite3.connect(BANCO_EVIDENCIAS) as con:
        return pd.read_sql_query(sql, con, params=params)

display(consultar_evidencias(
    'SELECT source_system, decisao, COUNT(*) AS quantidade FROM ev_record_decision '
    'WHERE dataset_id = ? AND rules_version = ? GROUP BY source_system, decisao ORDER BY source_system, decisao',
    (ID_DATASET, VERSAO_REGRAS)))
display(consultar_evidencias(
    'SELECT mes_id, sap_id, classe, temporal, subdivisao, componente, cadastro, duration_relevant '
    'FROM ev_pair_signal WHERE dataset_id = ? AND rules_version = ? ORDER BY mes_id, sap_id LIMIT 15',
    (ID_DATASET, VERSAO_REGRAS)))

## 8. Painel de parâmetros <a id="painel" name="painel"></a>

A janela, score e margem do painel abaixo pertencem à v3 e criam uma comparação isolada. O método candidato usa como padrão 60 min e já mostra sensibilidade em 30/60/120 min; esses parâmetros são propostas, não calibração. Nenhuma execução altera o banco anterior ou confirma vínculos.

In [ ]:
if EM_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()

janela = widgets.IntSlider(value=60, min=15, max=120, step=15, description='Janela min', continuous_update=False)
score = widgets.IntSlider(value=90, min=90, max=100, step=1, description='Score mínimo', continuous_update=False)
margem = widgets.IntSlider(value=10, min=10, max=30, step=5, description='Margem', continuous_update=False)
botao = widgets.Button(description='Recalcular e criar banco', button_style='primary',
                       layout=widgets.Layout(width='240px'))
saida = widgets.Output()
ULTIMO_BANCO = None


def associar_com_painel():
    """Reaplica o algoritmo sobre os dados já normalizados, com a política do painel."""
    experimento = copy.deepcopy(DADOS)
    ASSOCIACAO.POLICY.update(candidateWindowMinutes=janela.value, autoMinimumScore=score.value,
                             minimumWinnerMargin=margem.value)
    experimento['association'] = copy.deepcopy(ASSOCIACAO.associate(experimento['mes'], experimento['sap']))
    experimento['metadata']['experiment'] = 'Parâmetros ajustados no Colab; não publicado no SIME.'
    return experimento


def recalcular(_=None):
    global ULTIMO_BANCO
    with saida:
        saida.clear_output(wait=True)
        botao.disabled = True
        politica_original = copy.deepcopy(ASSOCIACAO.POLICY)
        try:
            experimento = associar_com_painel()
            fonte = PASTA / 'saidas' / f'experimento_{uuid4().hex[:8]}.json'
            fonte.write_text(json.dumps(experimento, ensure_ascii=False, separators=(',', ':')), encoding='utf8')
            ULTIMO_BANCO = fonte.with_suffix('.sqlite')
            display(pd.DataFrame([{'status': k, 'quantidade': v}
                                  for k, v in experimento['association']['summary'].items()]))
            display(BASE.create_database(fonte, ULTIMO_BANCO))
            print('Banco:', ULTIMO_BANCO.name)
        finally:
            # A política global volta ao padrão mesmo se o recálculo falhar.
            ASSOCIACAO.POLICY.clear()
            ASSOCIACAO.POLICY.update(politica_original)
            botao.disabled = False


botao.on_click(recalcular)
display(widgets.VBox([janela, score, margem, botao, saida]))

## 9. Testes <a id="testes" name="testes"></a>

Rodam testes unitários da v3 e de `sp2-evidence-v1`. Os testes de evidência são sintéticos: cobrem par provável pendente, disputa, ausência em cada fonte, duração, duplicidade/fragmentação, intervalos incompletos, equivalência do índice e determinismo; não medem acurácia industrial.

In [ ]:
teste = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(CONC),
                        '-p', 'test_associacao_automatica.py', '-v'],
                       capture_output=True, text=True, encoding='utf8', errors='replace')
print(teste.stderr)  # o unittest escreve o relatório em stderr
teste.check_returncode()

teste_evidencias = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(CONC),
                                   '-p', 'test_conciliacao_evidencias.py', '-v'],
                                  capture_output=True, text=True, encoding='utf8', errors='replace')
print(teste_evidencias.stderr)
teste_evidencias.check_returncode()

In [ ]:
mes = {'id': 'MES-DEMO', 'machine': 'A', 'subdivision': 'LONAS', 'component': 'LONA',
       'start': '2026-01-01T10:00:00', 'end': '2026-01-01T10:30:00'}
sap = {'id': 'SAP-DEMO', 'machine': 'A', 'subdivision': 'LONAS', 'text': 'LONA TRAVADA',
       'start': mes['start'], 'end': mes['end'], 'assetMappingValidated': False}
sap_validado = {**sap, 'assetMappingValidated': True, 'mappingStatus': 'validação sintética de exercício'}

cenarios = {
    'mapa inferido': ASSOCIACAO.associate([mes], [sap]),
    'mapa validado (sintético)': ASSOCIACAO.associate([mes], [sap_validado]),
    'duas ordens idênticas': ASSOCIACAO.associate([mes], [sap_validado, {**sap_validado, 'id': 'SAP-DEMO-2'}]),
}
assert cenarios['mapa inferido']['decisions'][0]['selectedSapId'] is None
assert cenarios['mapa validado (sintético)']['decisions'][0]['selectedSapId'] == 'SAP-DEMO'
assert cenarios['duas ordens idênticas']['decisions'][0]['selectedSapId'] is None
display(pd.DataFrame([{'cenário': nome, **r['decisions'][0]} for nome, r in cenarios.items()]).drop(columns=['candidates']))

## 10. Entrega <a id="entrega" name="entrega"></a>

O ZIP reúne banco, experimentos do painel, JSON normalizado, auditoria e hashes dos módulos. Ele pode conter textos da planilha; revise antes de compartilhar.

In [ ]:
def empacotar_entrega():
    manifesto = PASTA / 'saidas/modulos_sha256.json'
    manifesto.write_text(json.dumps(SOURCE_HASHES, indent=2), encoding='utf8')
    pacote = PASTA / f'entrega_mes_sap_{uuid4().hex[:8]}.zip'
    with zipfile.ZipFile(pacote, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for caminho in sorted((PASTA / 'saidas').glob('*')):
            if caminho.is_file():
                z.write(caminho, 'saidas/' + caminho.name)
        for nome in ['MES.csv', 'SAP.csv', 'inventario.json', 'perfil_qualidade.json', 'resumo.json']:
            z.write(AUDITOR.OUT / nome, 'auditoria/' + nome)
    return pacote


PACOTE = empacotar_entrega()
print('Pacote:', PACOTE, '| banco inicial:', BANCO.name, '| último do painel:', ULTIMO_BANCO and ULTIMO_BANCO.name)
if EM_COLAB:
    from google.colab import files
    files.download(str(PACOTE))

## 11. Limites <a id="limites" name="limites"></a>

- Dados `demo` são fictícios e não representam a planilha real.
- O mapa de ativos SAP é inferido; o notebook não o valida.
- `sp2-evidence-v1` é método candidato paralelo à v3; suas classes são ordinais, seus limiares são propostas e não há revisão humana registrada.
- Os totais de saída são indicadores independentes e podem se sobrepor: por exemplo, um registro pode ter um par provável e também uma divergência de duração ou um alerta de qualidade.
- Correspondência provável significa provável segundo as regras; não confirma que os registros representam o mesmo evento físico.
- Medir correção exige rótulos revisados por especialista; cobertura, velocidade e estabilidade de limiares não são acurácia.
- A linha do tempo compara intervalos registrados por sistema. Duração SAP não é tempo efetivo de reparo ou mão de obra; lacunas e intervalos incompletos permanecem visíveis.
- Recalcular não é conexão contínua com MES/SAP nem atualiza o SIME publicado ou o Azure.
- MTBF/MTTR, custo e causa raiz não saem só de juntar tabelas.